In [1]:
!cp -r /kaggle/input/notebooks/ivanesin/flood-map-v10/. /kaggle/working/

In [2]:
!pip install -q osmnx

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.4/104.4 kB 4.6 MB/s eta 0:00:00


In [3]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input/notebooks/ivanesin/flood-map-v10/'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/notebooks/ivanesin/flood-map-v10/__results__.html
/kaggle/input/notebooks/ivanesin/flood-map-v10/__notebook__.ipynb
/kaggle/input/notebooks/ivanesin/flood-map-v10/__output__.json
/kaggle/input/notebooks/ivanesin/flood-map-v10/custom.css
/kaggle/input/notebooks/ivanesin/flood-map-v10/cache/1f9d4cdc6c54d72c39e0ea931b30760bb0e8ce05.json
/kaggle/input/notebooks/ivanesin/flood-map-v10/cache/b508622f76713149c206083aaefcf30ae472a46a.json
/kaggle/input/notebooks/ivanesin/flood-map-v10/cache/4f08624fb409224ffe720a05e7dfb73594b10394.json
/kaggle/input/notebooks/ivanesin/flood-map-v10/cache/337f03bd811a27c21c0fa3eaed435a3fee80f42c.json
/kaggle/input/notebooks/ivanesin/flood-map-v10/spb_dem/arcticdem.vrt
/kaggle/input/notebooks/ivanesin/flood-map-v10/spb_dem/dem_ellipsoidal.tif
/kaggle/input/notebooks/ivanesin/flood-map-v10/spb_dem/dem_egm2008.tif
/kaggle/input/notebooks/ivanesin/flood-map-v10/spb_dem/calibration_report.json
/kaggle/input/notebooks/ivanesin/flood-map-v10/spb_dem/dem_

In [4]:
#!/usr/bin/env python3
"""
Диагностика: почему 4 паспортные точки расходятся на 4.5 м.

Для каждой точки смотрим:
  1. Окно 21×21 пиксель (42×42 м) DEM в EGM2008 — есть ли NoData,
     какова дисперсия (узкая конструкция?), медиана.
  2. Ближайшие ICESat-2 ATL08 точки (k=30) и их dZ — DEM в этом
     участке корректен или сам шумный?
  3. Превращаем dem_egm в "псевдо-БС77" через простой shift
     по ICESat-2 + средняя поправка геоида и сравниваем с паспортом.

Выход:
  - gcp_diagnostic.png:  4 панели с DEM-окнами и маркерами
  - gcp_diagnostic.json: численные результаты
  - терминал: подробный отчёт по каждой точке

Запуск:
    python diagnose_gcps.py
"""

from __future__ import annotations

import json
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.windows import Window
import matplotlib.pyplot as plt
from pyproj import Transformer
from scipy.spatial import cKDTree


# --- Что смотрим -------------------------------------------------------------
WORK_DIR = Path("./spb_dem")
DEM_EGM = WORK_DIR / "dem_egm2008.tif"
DEM_BS77 = WORK_DIR / "dem_bs77.tif"
ATL08_GPKG = WORK_DIR / "atl08_ground.gpkg"
OUT_PNG = WORK_DIR / "gcp_diagnostic.png"
OUT_JSON = WORK_DIR / "gcp_diagnostic.json"

GCPS = [
    ("Gorny_embankment", 30.269963, 59.929175, 2.846),
    ("KZS_01_V1",        29.694830, 59.994314, 4.347),
    ("KZS_02_S1",        29.696649, 59.987700, 3.367),
    ("KZS_03_S2_north",  29.836873, 60.027940, 3.770),
]

WIN_PX = 21          # окно 42×42 м для статистики
NEIGHBOR_K = 30      # сколько ближайших ATL08 точек
NEIGHBOR_MAX_M = 1000  # ограничение по дальности (м)


# =============================================================================

def diagnose_one(name: str, lon: float, lat: float, h_passport: float,
                 src: rasterio.DatasetReader,
                 atl08: gpd.GeoDataFrame, atl_tree: cKDTree) -> dict:
    """Полная диагностика одной паспортной точки."""
    tr = Transformer.from_crs(4326, src.crs, always_xy=True)
    x, y = tr.transform(lon, lat)

    # --- DEM окно вокруг точки ----------------------------------------------
    col = (x - src.transform.c) / src.transform.a
    row = (y - src.transform.f) / src.transform.e
    c0 = int(round(col)) - WIN_PX // 2
    r0 = int(round(row)) - WIN_PX // 2
    win = Window(c0, r0, WIN_PX, WIN_PX)
    tile = src.read(1, window=win, boundless=True,
                    fill_value=np.nan).astype(float)
    if src.nodata is not None:
        tile[tile == src.nodata] = np.nan
    valid = tile[~np.isnan(tile)]
    center = float(tile[WIN_PX // 2, WIN_PX // 2]) if valid.size else float("nan")

    # --- Ближайшие ICESat-2 точки -------------------------------------------
    dists, idx = atl_tree.query([x, y], k=NEIGHBOR_K)
    keep = dists <= NEIGHBOR_MAX_M
    if not keep.any():
        nearby = atl08.iloc[[]]
        atl_summary = {"n": 0}
    else:
        nearby = atl08.iloc[idx[keep]]
        # h_te_ell — эллипсоидальная высота от ICESat-2
        # h_dem_ell — DEM эллипсоидальный в той же точке
        # dZ = atl - dem ≈ 0 если DEM правильный
        atl_summary = {
            "n": int(keep.sum()),
            "min_dist_m": float(dists[keep].min()),
            "max_dist_m": float(dists[keep].max()),
            "h_te_ell_med": float(nearby["h_te_ell"].median()),
            "h_dem_ell_med": float(nearby["h_dem_ell"].median()),
            "dZ_med": float(nearby["dZ"].median()),
            "dZ_mad": float((nearby["dZ"] - nearby["dZ"].median()).abs().median()),
        }

    # --- Превратим паспорт в EGM2008 для сравнения --------------------------
    # БС-77 в районе Кронштадта ≈ EGM2008 + ~0.5 м (Балтийский футшток)
    # Это нестрогая оценка, но для диагностики хватает.
    # Если паспорт=2.846 БС-77, то ожидаемый EGM2008 ≈ 2.346
    h_passport_egm_approx = h_passport - 0.5

    # Сравнение
    delta_center = center - h_passport_egm_approx
    delta_median = (float(np.nanmedian(valid)) - h_passport_egm_approx
                    if valid.size else float("nan"))

    # --- Диагностические флаги ----------------------------------------------
    flags = []
    nan_frac = 1 - valid.size / tile.size
    if nan_frac > 0.3:
        flags.append(f"NoData в {100*nan_frac:.0f}% окна — возможно вода/край")
    if valid.size and (valid.max() - valid.min()) > 5:
        flags.append(f"высокая дисперсия в окне ({valid.max()-valid.min():.1f} м)"
                     " — узкая конструкция/здание")
    if valid.size and abs(np.nanmedian(valid)) < 0.5:
        flags.append("медиана окна ~0 м EGM2008 — похоже на воду "
                     "(Copernicus GLO-30 fill)")
    if atl_summary.get("n", 0) > 0 and abs(atl_summary["dZ_med"]) > 1.0:
        flags.append(f"ICESat-2 в окрестности расходится с DEM на "
                     f"{atl_summary['dZ_med']:+.2f} м — DEM локально шумный")
    if atl_summary.get("n", 0) > 0 and abs(atl_summary["dZ_med"]) <= 0.5:
        flags.append("ICESat-2 локально подтверждает значение DEM "
                     "(DEM здесь правильный)")
    if abs(delta_median) > 1.5:
        flags.append(f"паспорт расходится с медианой DEM в окне на "
                     f"{delta_median:+.2f} м → проверьте координаты или высоту")

    print(f"\n{'=' * 60}")
    print(f"  {name}")
    print(f"{'=' * 60}")
    print(f"  Координаты: lat={lat:.6f}, lon={lon:.6f}")
    print(f"  Паспорт:    {h_passport:+.3f} м БС-77 "
          f"(≈ {h_passport_egm_approx:+.2f} м EGM2008)")
    print(f"  DEM окно 42×42 м:")
    print(f"    центр пикселя:  {center:+.3f} м EGM")
    if valid.size:
        print(f"    медиана:        {np.nanmedian(valid):+.3f}")
        print(f"    min..max:       {valid.min():+.2f} .. {valid.max():+.2f}")
        print(f"    std:            {np.nanstd(valid):.3f}")
        print(f"    валидных:       {valid.size}/{tile.size} "
              f"({100*valid.size/tile.size:.0f}%)")
    print(f"  ICESat-2 ближайшие в радиусе ≤{NEIGHBOR_MAX_M} м:")
    if atl_summary["n"]:
        print(f"    точек:          {atl_summary['n']}")
        print(f"    дистанция:      {atl_summary['min_dist_m']:.0f} .. "
              f"{atl_summary['max_dist_m']:.0f} м")
        print(f"    h_ATL08 медиана:    {atl_summary['h_te_ell_med']:+.2f} м эллипс")
        print(f"    h_DEM медиана:      {atl_summary['h_dem_ell_med']:+.2f} м эллипс")
        print(f"    dZ = ATL−DEM:       {atl_summary['dZ_med']:+.2f} ± "
              f"{atl_summary['dZ_mad']:.2f} м")
    else:
        print(f"    нет точек в радиусе {NEIGHBOR_MAX_M} м")
    if flags:
        print(f"  ⚠ Флаги:")
        for f in flags:
            print(f"     • {f}")
    else:
        print(f"  ✓ диагностика чистая")

    return {
        "name": name,
        "passport_bs77": h_passport,
        "passport_egm_approx": round(h_passport_egm_approx, 3),
        "dem_center_egm": round(center, 3) if not np.isnan(center) else None,
        "dem_window": ({
            "median": round(float(np.nanmedian(valid)), 3),
            "min": round(float(valid.min()), 3),
            "max": round(float(valid.max()), 3),
            "std": round(float(np.nanstd(valid)), 3),
            "n_valid": int(valid.size),
            "n_total": int(tile.size),
            "nan_fraction": round(float(nan_frac), 3),
        } if valid.size else None),
        "delta_center_m": round(delta_center, 3) if not np.isnan(delta_center) else None,
        "delta_median_m": round(delta_median, 3) if not np.isnan(delta_median) else None,
        "icesat2_neighborhood": atl_summary,
        "flags": flags,
    }, tile


def main() -> None:
    if not DEM_EGM.exists():
        raise SystemExit(f"Нет {DEM_EGM}")
    if not ATL08_GPKG.exists():
        raise SystemExit(f"Нет {ATL08_GPKG}")

    print(f"Читаю ATL08 точки из {ATL08_GPKG}...")
    atl08 = gpd.read_file(ATL08_GPKG, layer="atl08_with_dem")
    # Колонки x_m, y_m уже в WORK_EPSG=32636 (см. spb_dem_build_v2.py)
    print(f"  загружено: {len(atl08):,}")

    atl_tree = cKDTree(atl08[["x_m", "y_m"]].values)

    src = rasterio.open(DEM_EGM)
    print(f"DEM CRS: {src.crs}, size: {src.width}×{src.height}")

    results = []
    tiles = []
    for name, lon, lat, h in GCPS:
        rec, tile = diagnose_one(name, lon, lat, h, src, atl08, atl_tree)
        results.append(rec)
        tiles.append((name, h, tile, rec))

    src.close()

    # --- Визуализация --------------------------------------------------------
    fig, axes = plt.subplots(2, 2, figsize=(12, 12))
    for ax, (name, h, tile, rec) in zip(axes.flatten(), tiles):
        if not np.isnan(tile).all():
            vmin = float(np.nanpercentile(tile, 2))
            vmax = float(np.nanpercentile(tile, 98))
        else:
            vmin, vmax = 0, 5
        im = ax.imshow(tile, cmap="terrain", vmin=vmin, vmax=vmax)
        ax.plot(WIN_PX // 2, WIN_PX // 2, marker="*", markersize=22,
                color="red", markeredgecolor="white", markeredgewidth=2)
        title = (f"{name}\nпаспорт: {h:+.2f} БС-77  "
                 f"центр: {rec['dem_center_egm']:+.2f} EGM\n"
                 f"медиана 42м: {rec['dem_window']['median']:+.2f} EGM"
                 if rec['dem_window'] else f"{name}\nNoData")
        ax.set_title(title, fontsize=10)
        ax.set_xlabel("колонка пикселя")
        ax.set_ylabel("строка пикселя")
        plt.colorbar(im, ax=ax, label="м EGM2008", shrink=0.8)
    fig.suptitle("Диагностика паспортных точек: окно 42×42 м, "
                 "красная звезда — координаты GPS", fontsize=12)
    fig.tight_layout()
    fig.savefig(OUT_PNG, dpi=130, bbox_inches="tight")
    plt.close(fig)
    print(f"\n✓ Карта окон сохранена: {OUT_PNG}")

    # --- JSON ----------------------------------------------------------------
    OUT_JSON.write_text(json.dumps(
        {"gcp_diagnostics": results}, indent=2, ensure_ascii=False))
    print(f"✓ JSON отчёт:        {OUT_JSON}")

    # --- Сводная таблица -----------------------------------------------------
    print(f"\n{'=' * 60}")
    print(f"  СВОДКА")
    print(f"{'=' * 60}")
    print(f"  {'Точка':<22} {'Δ_центр':>10} {'Δ_мед':>10} {'ICESat-2 dZ':>14}")
    for r in results:
        atl = r["icesat2_neighborhood"]
        atl_str = f"{atl.get('dZ_med', 0):+.2f}" if atl.get("n", 0) else "n/a"
        dc = f"{r['delta_center_m']:+.2f}" if r['delta_center_m'] is not None else "—"
        dm = f"{r['delta_median_m']:+.2f}" if r['delta_median_m'] is not None else "—"
        print(f"  {r['name']:<22} {dc:>10} {dm:>10} {atl_str:>14}")
    print(f"\n  Δ — паспорт_egm − DEM_egm")
    print(f"  ICESat-2 dZ — h_ATL − h_DEM в окрестности (≈0 = DEM в этом районе ОК)")


if __name__ == "__main__":
    main()

Читаю ATL08 точки из spb_dem/atl08_ground.gpkg...
  загружено: 43,334
DEM CRS: EPSG:32636, size: 52224×33895

  Gorny_embankment
  Координаты: lat=59.929175, lon=30.269963
  Паспорт:    +2.846 м БС-77 (≈ +2.35 м EGM2008)
  DEM окно 42×42 м:
    центр пикселя:  +3.678 м EGM
    медиана:        +3.356
    min..max:       +1.58 .. +17.74
    std:            3.876
    валидных:       441/441 (100%)
  ICESat-2 ближайшие в радиусе ≤1000 м:
    точек:          30
    дистанция:      136 .. 1000 м
    h_ATL08 медиана:    +18.83 м эллипс
    h_DEM медиана:      +19.98 м эллипс
    dZ = ATL−DEM:       -1.25 ± 1.61 м
  ⚠ Флаги:
     • высокая дисперсия в окне (16.2 м) — узкая конструкция/здание
     • ICESat-2 в окрестности расходится с DEM на -1.25 м — DEM локально шумный

  KZS_01_V1
  Координаты: lat=59.994314, lon=29.694830
  Паспорт:    +4.347 м БС-77 (≈ +3.85 м EGM2008)
  DEM окно 42×42 м:
    центр пикселя:  +2.856 м EGM
    медиана:        +2.557
    min..max:       +2.11 .. +2.96
    std

In [5]:
#!/usr/bin/env python3
"""
Snap GCPs к ближайшему плоскому пикселю DEM с правильной высотой.

Идея:
  Для каждой паспортной точки (lat, lon, h_passport) ищем в радиусе
  SEARCH_RADIUS_M пиксель P, который удовлетворяет:
    A) ОЖИДАЕМАЯ ВЫСОТА: |dem(P) - (h_passport - 0.5)| <= H_TOL_M
    B) ПЛОСКИЙ:           std(окна WIN×WIN вокруг P) <= STD_TOL_M
    C) ПОДДЕРЖКА ICESat-2: есть ATL08 точка в радиусе ATL_R_M с малым dZ
    D) МИНИМИЗИРУЕМ суммарную метрику = α·|Δh| + β·std + γ·dist

  В случае нескольких кандидатов выбираем самого «уверенного» по метрике.
  Если для точки кандидатов нет — печатаем диагностику и предлагаем
  расширить радиус или признать, что точку надо проверить вручную.

Выход:
  - snapped_gcps.json: новые координаты + метрики снапа
  - snapped_gcps.png:  для каждой точки — окно DEM с двумя маркерами:
                       красная звезда = исходная Yandex-координата,
                       зелёный круг   = найденная snap-позиция

Запуск:
    python snap_gcps.py
"""

from __future__ import annotations

import json
from pathlib import Path

import numpy as np
import geopandas as gpd
import rasterio
from rasterio.windows import Window
import matplotlib.pyplot as plt
from pyproj import Transformer
from scipy.ndimage import generic_filter
from scipy.spatial import cKDTree


# --- Конфигурация ------------------------------------------------------------
WORK_DIR = Path("./spb_dem")
DEM_EGM = WORK_DIR / "dem_egm2008.tif"
ATL08_GPKG = WORK_DIR / "atl08_ground.gpkg"

OUT_JSON = WORK_DIR / "snapped_gcps.json"
OUT_PNG = WORK_DIR / "snapped_gcps.png"

# Паспортные точки (Yandex/Google координаты, могут лежать на воде)
GCPS = [
    ("Gorny_embankment", 30.269963, 59.929175, 2.846),
    ("KZS_01_V1",        29.694830, 59.994314, 4.347),
    ("KZS_02_S1",        29.696649, 59.987700, 3.367),
    ("KZS_03_S2_north",  29.836873, 60.027940, 3.770),
]

# --- Параметры snap ----------------------------------------------------------
SEARCH_RADIUS_M = 500        # радиус поиска вокруг исходной точки
H_TOL_M = 1.5                # допуск по высоте (паспорт - 0.5 - DEM_egm)
STD_TOL_M = 0.30             # допуск по плоскости (std окна)
WIN_FLAT = 5                 # окно для оценки плоскости (5×5 пикс = 10×10 м)
ATL_R_M = 500                # радиус поиска ATL08 для подтверждения
ATL_DZ_TOL = 1.5             # |dZ| ≤ — для подтверждения

# Веса метрики выбора лучшего кандидата
W_DH = 1.0       # вес отклонения по высоте (м)
W_STD = 2.0      # вес шума (м) — плоскость важнее всего
W_DIST = 0.005   # вес дальности от исходной точки (м)
                 # (0.005 — мягко: 100 м прибавляют 0.5 к метрике)


# =============================================================================

def egm_target(passport_bs77: float) -> float:
    """Перевод БС-77 → ожидаемая высота в EGM2008. Кронштадтский анкер ≈ +0.5."""
    return passport_bs77 - 0.5


def find_snap_for_point(name: str, lon: float, lat: float, h_passport: float,
                        src: rasterio.DatasetReader,
                        atl_xy: np.ndarray, atl_dz: np.ndarray,
                        atl_tree: cKDTree) -> dict:
    """Ищем лучшую snap-позицию в радиусе вокруг (lon, lat)."""
    tr = Transformer.from_crs(4326, src.crs, always_xy=True)
    cx, cy = tr.transform(lon, lat)

    # Окно поиска (квадрат +-R с запасом для расчёта std)
    px = src.transform.a   # размер пикселя по X (м)
    py = -src.transform.e  # по Y
    half_search_px = int(np.ceil(SEARCH_RADIUS_M / max(px, py))) + WIN_FLAT
    col_c = (cx - src.transform.c) / src.transform.a
    row_c = (cy - src.transform.f) / src.transform.e
    c0 = int(round(col_c)) - half_search_px
    r0 = int(round(row_c)) - half_search_px
    win_size = 2 * half_search_px + 1
    win = Window(c0, r0, win_size, win_size)

    tile = src.read(1, window=win, boundless=True,
                    fill_value=np.nan).astype(float)
    if src.nodata is not None:
        tile[tile == src.nodata] = np.nan

    # Маска валидных + посчитать std в скользящем окне WIN_FLAT
    def _local_std(arr):
        v = arr[~np.isnan(arr)]
        return float(np.std(v)) if v.size >= arr.size * 0.5 else np.nan

    std_map = generic_filter(tile, _local_std, size=WIN_FLAT, mode="constant",
                             cval=np.nan)

    # Координаты центра каждого пикселя в CRS растра
    rr, cc = np.mgrid[r0:r0 + win_size, c0:c0 + win_size]
    xs = src.transform.c + (cc + 0.5) * src.transform.a
    ys = src.transform.f + (rr + 0.5) * src.transform.e
    dist_m = np.hypot(xs - cx, ys - cy)

    # Маска радиуса поиска
    in_radius = dist_m <= SEARCH_RADIUS_M

    # --- Ограничение A: по высоте --------------------------------------------
    target_h = egm_target(h_passport)
    dh = tile - target_h
    mask_h = np.abs(dh) <= H_TOL_M

    # --- Ограничение B: плоскость --------------------------------------------
    mask_flat = std_map <= STD_TOL_M

    # --- Объединить ----------------------------------------------------------
    valid = in_radius & mask_h & mask_flat & ~np.isnan(tile)

    n_in_radius = int((in_radius & ~np.isnan(tile)).sum())
    n_pass_h = int((in_radius & mask_h & ~np.isnan(tile)).sum())
    n_pass_flat = int((in_radius & mask_flat & ~np.isnan(tile)).sum())
    n_pass_both = int(valid.sum())

    print(f"\n{'=' * 60}")
    print(f"  {name}  (паспорт {h_passport:+.2f} БС-77, "
          f"target EGM = {target_h:+.2f} ± {H_TOL_M})")
    print(f"{'=' * 60}")
    print(f"  Поиск в радиусе {SEARCH_RADIUS_M} м: {n_in_radius} валидных пикс")
    print(f"  Прошло по высоте (|Δh|≤{H_TOL_M}):    {n_pass_h}")
    print(f"  Прошло по плоскости (std≤{STD_TOL_M}): {n_pass_flat}")
    print(f"  Прошло по обоим:                     {n_pass_both}")

    if n_pass_both == 0:
        print(f"  ✗ Кандидаты не найдены. Расширьте SEARCH_RADIUS_M, "
              f"H_TOL_M или STD_TOL_M.")
        return {
            "name": name, "snap_found": False,
            "orig_lon": lon, "orig_lat": lat,
            "diagnostics": {"n_in_radius": n_in_radius,
                            "n_pass_h": n_pass_h,
                            "n_pass_flat": n_pass_flat},
            "tile": tile, "tile_origin": (c0, r0, src),
            "orig_xy": (cx, cy),
        }

    # --- Подтверждение по ICESat-2 -------------------------------------------
    # Для каждого кандидата ищем ATL08 в радиусе ATL_R_M
    cand_idx = np.argwhere(valid)
    cand_xs = xs[valid]
    cand_ys = ys[valid]
    n_cands = len(cand_xs)

    atl_support = np.zeros(n_cands, dtype=bool)
    atl_dz_med = np.full(n_cands, np.nan)

    if len(atl_xy) > 0:
        # Запрос ATL08 точек в радиусе для каждого кандидата
        for i, (xq, yq) in enumerate(zip(cand_xs, cand_ys)):
            ids = atl_tree.query_ball_point([xq, yq], r=ATL_R_M)
            if len(ids) >= 3:
                dz = atl_dz[ids]
                atl_dz_med[i] = float(np.median(dz))
                if abs(atl_dz_med[i]) <= ATL_DZ_TOL:
                    atl_support[i] = True

    # --- Метрика и выбор лучшего --------------------------------------------
    cand_dh = np.abs(dh[valid])
    cand_std = std_map[valid]
    cand_dist = dist_m[valid]
    metric = (W_DH * cand_dh
              + W_STD * cand_std
              + W_DIST * cand_dist
              + np.where(atl_support, 0.0, 0.5))  # бонус за поддержку ICESat-2

    best = int(np.argmin(metric))

    best_row, best_col = cand_idx[best]   # индексы в локальном tile
    best_x = cand_xs[best]; best_y = cand_ys[best]
    best_lon, best_lat = Transformer.from_crs(
        src.crs, 4326, always_xy=True).transform(best_x, best_y)

    n_supported = int(atl_support.sum())
    print(f"  ICESat-2 подтвердил {n_supported}/{n_cands} кандидатов")
    print(f"  Выбран лучший:")
    print(f"    координата:    lat={best_lat:.6f}, lon={best_lon:.6f}")
    print(f"    смещение:      {cand_dist[best]:.1f} м от исходной")
    print(f"    DEM:           {tile[best_row, best_col]:+.3f} м EGM "
          f"(target {target_h:+.2f}, Δ={cand_dh[best]:+.2f})")
    print(f"    std окна:      {cand_std[best]:.3f} м")
    if atl_support[best]:
        print(f"    ICESat-2:      ✓ dZ_med = {atl_dz_med[best]:+.2f} м")
    else:
        if not np.isnan(atl_dz_med[best]):
            print(f"    ICESat-2:      ⚠ dZ_med = {atl_dz_med[best]:+.2f} м "
                  f"(>{ATL_DZ_TOL})")
        else:
            print(f"    ICESat-2:      нет точек в радиусе {ATL_R_M} м")
    print(f"    metric:        {metric[best]:.3f}")

    return {
        "name": name, "snap_found": True,
        "orig_lon": lon, "orig_lat": lat,
        "snap_lon": float(best_lon), "snap_lat": float(best_lat),
        "snap_distance_m": round(float(cand_dist[best]), 2),
        "passport_bs77": h_passport,
        "target_egm": round(target_h, 3),
        "dem_at_snap_egm": round(float(tile[best_row, best_col]), 3),
        "delta_h_m": round(float(cand_dh[best]), 3),
        "window_std_m": round(float(cand_std[best]), 3),
        "icesat2_support": bool(atl_support[best]),
        "icesat2_dz_med_m": (round(float(atl_dz_med[best]), 3)
                             if not np.isnan(atl_dz_med[best]) else None),
        "n_candidates": n_cands,
        "n_icesat2_supported": n_supported,
        "metric": round(float(metric[best]), 3),
        "tile": tile, "tile_origin": (c0, r0, src),
        "orig_xy": (cx, cy), "best_xy": (best_x, best_y),
    }


def main() -> None:
    print(f"Читаю ATL08 точки из {ATL08_GPKG}...")
    atl08 = gpd.read_file(ATL08_GPKG, layer="atl08_with_dem")
    atl_xy = atl08[["x_m", "y_m"]].values
    atl_dz = atl08["dZ"].values
    atl_tree = cKDTree(atl_xy)
    print(f"  загружено: {len(atl08):,}")

    src = rasterio.open(DEM_EGM)
    print(f"DEM CRS: {src.crs}, size: {src.width}×{src.height}")

    results = []
    for name, lon, lat, h in GCPS:
        rec = find_snap_for_point(name, lon, lat, h, src,
                                  atl_xy, atl_dz, atl_tree)
        results.append(rec)

    src.close()

    # --- Визуализация --------------------------------------------------------
    fig, axes = plt.subplots(2, 2, figsize=(13, 13))
    for ax, rec in zip(axes.flatten(), results):
        tile = rec["tile"]
        c0, r0, src_ref = rec["tile_origin"]
        if not np.isnan(tile).all():
            vmin = float(np.nanpercentile(tile, 2))
            vmax = float(np.nanpercentile(tile, 98))
        else:
            vmin, vmax = 0, 5
        im = ax.imshow(tile, cmap="terrain", vmin=vmin, vmax=vmax)

        # Исходная точка
        cx, cy = rec["orig_xy"]
        # Преобразуем в локальные пиксельные координаты tile
        pcol = (cx - src_ref.transform.c) / src_ref.transform.a - c0 - 0.5
        prow = (cy - src_ref.transform.f) / src_ref.transform.e - r0 - 0.5
        ax.plot(pcol, prow, marker="*", markersize=22, color="red",
                markeredgecolor="white", markeredgewidth=2,
                label="исходная (Yandex)")

        # Snap-позиция
        if rec.get("snap_found"):
            bx, by = rec["best_xy"]
            scol = (bx - src_ref.transform.c) / src_ref.transform.a - c0 - 0.5
            srow = (by - src_ref.transform.f) / src_ref.transform.e - r0 - 0.5
            ax.plot(scol, srow, marker="o", markersize=18, color="lime",
                    markeredgecolor="darkgreen", markeredgewidth=3,
                    label=f"snap (Δ={rec['snap_distance_m']:.0f}м)")
            title = (f"{rec['name']}\n"
                     f"target {rec['target_egm']:+.2f} → "
                     f"snap DEM={rec['dem_at_snap_egm']:+.2f}, "
                     f"std={rec['window_std_m']:.2f}\n"
                     f"ICESat-2: {'✓' if rec['icesat2_support'] else '✗'}")
        else:
            title = f"{rec['name']}\n✗ кандидаты не найдены"

        ax.set_title(title, fontsize=10)
        ax.legend(loc="upper right", fontsize=8)
        ax.set_xlabel("колонка пикс (от tile)")
        ax.set_ylabel("строка пикс")
        plt.colorbar(im, ax=ax, label="м EGM2008", shrink=0.8)
    fig.suptitle(
        f"Snap-to-flat-DEM: исходная Yandex (★) → snap-позиция (●) "
        f"[радиус {SEARCH_RADIUS_M}м, |Δh|≤{H_TOL_M}м, std≤{STD_TOL_M}м]",
        fontsize=11)
    fig.tight_layout()
    fig.savefig(OUT_PNG, dpi=130, bbox_inches="tight")
    plt.close(fig)
    print(f"\n✓ Карта снапов: {OUT_PNG}")

    # --- JSON для сохранения новых координат ---------------------------------
    serializable = []
    for r in results:
        out = {k: v for k, v in r.items()
               if k not in ("tile", "tile_origin", "orig_xy", "best_xy")}
        serializable.append(out)
    OUT_JSON.write_text(json.dumps(
        {"snap_results": serializable,
         "config": {
             "search_radius_m": SEARCH_RADIUS_M,
             "h_tol_m": H_TOL_M,
             "std_tol_m": STD_TOL_M,
             "win_flat_px": WIN_FLAT,
             "atl_r_m": ATL_R_M,
         }}, indent=2, ensure_ascii=False))
    print(f"✓ JSON:         {OUT_JSON}")

    # --- Сводка с готовым кодом для копирования в spb_dem_build_v2.py -------
    print(f"\n{'=' * 60}")
    print(f"  СВОДКА: новые координаты для GCPS в spb_dem_build_v2.py")
    print(f"{'=' * 60}")
    for r in results:
        if r.get("snap_found"):
            print(f"    GCP(\"{r['name']}\",")
            print(f"        lon={r['snap_lon']:.6f}, lat={r['snap_lat']:.6f}, "
                  f"h_bs77={r['passport_bs77']},")
            print(f"        source=\"snapped (Δ={r['snap_distance_m']:.1f}м "
                  f"от Yandex, std={r['window_std_m']:.2f}м)\"),")
        else:
            print(f"    # {r['name']}: snap не найден — проверить вручную")


if __name__ == "__main__":
    main()

Читаю ATL08 точки из spb_dem/atl08_ground.gpkg...
  загружено: 43,334
DEM CRS: EPSG:32636, size: 52224×33895

  Gorny_embankment  (паспорт +2.85 БС-77, target EGM = +2.35 ± 1.5)
  Поиск в радиусе 500 м: 196359 валидных пикс
  Прошло по высоте (|Δh|≤1.5):    62359
  Прошло по плоскости (std≤0.3): 89328
  Прошло по обоим:                     56038
  ICESat-2 подтвердил 17037/56038 кандидатов
  Выбран лучший:
    координата:    lat=59.929320, lon=30.270442
    смещение:      31.3 м от исходной
    DEM:           +2.348 м EGM (target +2.35, Δ=+0.00)
    std окна:      0.134 м
    ICESat-2:      ⚠ dZ_med = -3.62 м (>1.5)
    metric:        0.926

  KZS_01_V1  (паспорт +4.35 БС-77, target EGM = +3.85 ± 1.5)
  Поиск в радиусе 500 м: 196369 валидных пикс
  Прошло по высоте (|Δh|≤1.5):    27058
  Прошло по плоскости (std≤0.3): 165547
  Прошло по обоим:                     16961
  ICESat-2 подтвердил 16363/16961 кандидатов
  Выбран лучший:
    координата:    lat=59.994272, lon=29.694922
    смещ

In [6]:
#!/usr/bin/env python3
"""
Snap v2: с правильным target = passport_BS77 − dEGM2008_BS77,
плюс независимая проверка через калибровку по воде.

Что меняется vs snap_gcps.py:
  1. Сдвиг EGM2008→БС-77 НЕ задаётся литературно (−0.5 был ошибкой).
     Вместо этого:
        - Пробуем несколько target-сдвигов (1.0, 1.2, 1.4, 1.6 м)
        - Для каждого делаем snap для KZS_01 и KZS_02 (надёжные точки)
        - Выбираем сдвиг, при котором snap для них даёт min(|Δh|)
        - Перепроверяем по водным точкам (см. ниже)

  2. Водная кросс-проверка:
        - В bbox КЗС берём water-mask (Copernicus fill даёт ~постоянный
          уровень моря в EGM2008)
        - Известное среднее БС-77 уровня Балтики у КЗС: ~−0.10 м
          (Кронштадтский футшток = средний многолетний уровень)
        - Невы у Горного: ~−0.05 м БС-77 в среднем (с поправкой на ветер)
        - Разница (вода_БС77) − (вода_EGM_DEM) = тот же сдвиг

  3. Исходные точки разделяются на:
        - Калибровочные (KZS_01, KZS_02) — устойчивые
        - Валидационные (KZS_03, Gorny) — для проверки

Запуск:
    python snap_gcps_v2.py
"""

from __future__ import annotations

import json
from pathlib import Path

import numpy as np
import geopandas as gpd
import rasterio
from rasterio.windows import Window
import matplotlib.pyplot as plt
from pyproj import Transformer
from scipy.ndimage import generic_filter
from scipy.spatial import cKDTree


WORK_DIR = Path("./spb_dem")
DEM_EGM = WORK_DIR / "dem_egm2008.tif"
ATL08_GPKG = WORK_DIR / "atl08_ground.gpkg"

OUT_JSON = WORK_DIR / "snapped_gcps_v2.json"
OUT_PNG = WORK_DIR / "snapped_gcps_v2.png"

# --- Точки разделены на role: calibration / validation ---------------------
GCPS = [
    # name,           lon,        lat,      h_bs77, role
    ("KZS_01_V1",       29.694830, 59.994314, 4.347, "calibration"),
    ("KZS_02_S1",       29.696649, 59.987700, 3.367, "calibration"),
    ("KZS_03_S2_north", 29.836873, 60.027940, 3.770, "validation"),
    ("Gorny_embankment", 30.269963, 59.929175, 2.846, "validation"),
]

# --- Параметры snap (как в v1) -----------------------------------------------
SEARCH_RADIUS_M = 100        # узкий радиус — берём только локальные кандидаты
H_TOL_M = 1.0                # уже H_TOL — после fix offset ожидаем плотную сходимость
STD_TOL_M = 0.30
WIN_FLAT = 5
ATL_R_M = 500
ATL_DZ_TOL = 1.5

W_DH = 1.0; W_STD = 2.0; W_DIST = 0.005

# --- Параметры поиска target offset ------------------------------------------
# Перебираем эти значения как кандидаты (passport - offset = target_egm).
# Из snap v1 ожидаем оптимум в районе 1.4.
OFFSET_GRID = np.arange(0.8, 2.21, 0.1)  # м

# --- Параметры водной калибровки ---------------------------------------------
# Известные средние уровни воды в БС-77 (приблизительно):
WATER_BS77_KZS = -0.10    # Финский залив у КЗС, средний многолетний
WATER_BS77_NEVA = -0.05   # Нева у Горного, средний межень

# Локальные bbox для выборки водных пикселей (lon_min, lat_min, lon_max, lat_max)
WATER_BBOXES = {
    "KZS_finzaliv": (29.65, 59.97, 29.85, 60.03),  # Финский залив у КЗС
    "Neva_at_Gorny": (30.25, 59.92, 30.29, 59.94),  # Нева у Горного
}


# =============================================================================
#                         ВОДНАЯ КАЛИБРОВКА (новое)
# =============================================================================

def estimate_water_levels(src: rasterio.DatasetReader) -> dict:
    """
    Оцениваем уровень воды в EGM2008 по двум зонам.
    Идея: Copernicus GLO-30 fill в ArcticDEM v4.1 даёт почти постоянное
    значение по водным пикселям ≈ EGM2008 уровень моря на момент GLO-30 съёмки.
    Берём медиану по большой акватории — это и есть оценка нуля воды.
    """
    tr = Transformer.from_crs(4326, src.crs, always_xy=True)
    results = {}

    for name, (lo1, la1, lo2, la2) in WATER_BBOXES.items():
        x1, y1 = tr.transform(lo1, la1)
        x2, y2 = tr.transform(lo2, la2)
        col1 = (min(x1, x2) - src.transform.c) / src.transform.a
        col2 = (max(x1, x2) - src.transform.c) / src.transform.a
        row1 = (max(y1, y2) - src.transform.f) / src.transform.e
        row2 = (min(y1, y2) - src.transform.f) / src.transform.e
        c_min, c_max = int(min(col1, col2)), int(max(col1, col2))
        r_min, r_max = int(min(row1, row2)), int(max(row1, row2))
        win = Window(c_min, r_min, c_max - c_min, r_max - r_min)
        tile = src.read(1, window=win, boundless=True,
                        fill_value=np.nan).astype(float)
        if src.nodata is not None:
            tile[tile == src.nodata] = np.nan

        # Водные пиксели = медиана-LOW значения (вода всегда ниже земли)
        # Простая эвристика: берём пиксели в диапазоне [-3, +1] м EGM
        # — это диапазон, в котором лежит water fill, не низкая земля.
        flat = tile[(tile >= -3) & (tile <= 1)]
        if flat.size < 100:
            print(f"  ⚠ {name}: мало водных пикселей ({flat.size})")
            results[name] = None
            continue
        med = float(np.median(flat))
        std = float(np.std(flat))
        # Узкое распределение std<0.3 — типично для GLO-30 water fill
        results[name] = {
            "n_pixels": int(flat.size),
            "median_egm": round(med, 3),
            "std_egm": round(std, 3),
        }

    return results


def water_calibration_offset(water_levels: dict) -> dict:
    """
    Сопоставляем медиану воды в EGM2008 с известным БС-77.
    offset = water_BS77 − water_EGM_median
    """
    out = {}
    if water_levels.get("KZS_finzaliv") is not None:
        wl = water_levels["KZS_finzaliv"]
        out["from_KZS_finzaliv"] = {
            "water_egm": wl["median_egm"],
            "water_bs77_assumed": WATER_BS77_KZS,
            "offset_egm_to_bs77": round(WATER_BS77_KZS - wl["median_egm"], 3),
        }
    if water_levels.get("Neva_at_Gorny") is not None:
        wl = water_levels["Neva_at_Gorny"]
        out["from_Neva_at_Gorny"] = {
            "water_egm": wl["median_egm"],
            "water_bs77_assumed": WATER_BS77_NEVA,
            "offset_egm_to_bs77": round(WATER_BS77_NEVA - wl["median_egm"], 3),
        }
    return out


# =============================================================================
#                    SNAP-функция (с настраиваемым offset)
# =============================================================================

def find_snap(name: str, lon: float, lat: float, h_passport: float,
              egm_to_bs77_offset: float, src: rasterio.DatasetReader,
              atl_xy: np.ndarray, atl_dz: np.ndarray, atl_tree: cKDTree,
              verbose: bool = True) -> dict:
    """target_egm = h_passport - offset_egm_to_bs77."""
    target_h = h_passport - egm_to_bs77_offset

    tr = Transformer.from_crs(4326, src.crs, always_xy=True)
    cx, cy = tr.transform(lon, lat)

    px = src.transform.a; py = -src.transform.e
    half_search_px = int(np.ceil(SEARCH_RADIUS_M / max(px, py))) + WIN_FLAT
    col_c = (cx - src.transform.c) / src.transform.a
    row_c = (cy - src.transform.f) / src.transform.e
    c0 = int(round(col_c)) - half_search_px
    r0 = int(round(row_c)) - half_search_px
    win_size = 2 * half_search_px + 1
    win = Window(c0, r0, win_size, win_size)

    tile = src.read(1, window=win, boundless=True,
                    fill_value=np.nan).astype(float)
    if src.nodata is not None:
        tile[tile == src.nodata] = np.nan

    def _local_std(arr):
        v = arr[~np.isnan(arr)]
        return float(np.std(v)) if v.size >= arr.size * 0.5 else np.nan

    std_map = generic_filter(tile, _local_std, size=WIN_FLAT, mode="constant",
                             cval=np.nan)
    rr, cc = np.mgrid[r0:r0 + win_size, c0:c0 + win_size]
    xs = src.transform.c + (cc + 0.5) * src.transform.a
    ys = src.transform.f + (rr + 0.5) * src.transform.e
    dist_m = np.hypot(xs - cx, ys - cy)
    in_radius = dist_m <= SEARCH_RADIUS_M

    dh = tile - target_h
    mask_h = np.abs(dh) <= H_TOL_M
    mask_flat = std_map <= STD_TOL_M
    valid = in_radius & mask_h & mask_flat & ~np.isnan(tile)

    if not valid.any():
        return {"name": name, "snap_found": False,
                "egm_offset_used": egm_to_bs77_offset,
                "target_egm": round(target_h, 3),
                "orig_lon": lon, "orig_lat": lat,
                "tile": tile, "tile_origin": (c0, r0, src),
                "orig_xy": (cx, cy)}

    cand_idx = np.argwhere(valid)
    cand_xs = xs[valid]; cand_ys = ys[valid]
    n_cands = len(cand_xs)
    atl_dz_med = np.full(n_cands, np.nan)
    atl_support = np.zeros(n_cands, dtype=bool)
    if len(atl_xy) > 0:
        for i, (xq, yq) in enumerate(zip(cand_xs, cand_ys)):
            ids = atl_tree.query_ball_point([xq, yq], r=ATL_R_M)
            if len(ids) >= 3:
                dz = atl_dz[ids]
                atl_dz_med[i] = float(np.median(dz))
                if abs(atl_dz_med[i]) <= ATL_DZ_TOL:
                    atl_support[i] = True

    cand_dh = np.abs(dh[valid])
    cand_std = std_map[valid]
    cand_dist = dist_m[valid]
    metric = (W_DH * cand_dh + W_STD * cand_std + W_DIST * cand_dist
              + np.where(atl_support, 0.0, 0.5))

    best = int(np.argmin(metric))
    bx, by = cand_xs[best], cand_ys[best]
    bl, bla = Transformer.from_crs(src.crs, 4326, always_xy=True).transform(bx, by)
    best_row, best_col = cand_idx[best]

    return {
        "name": name, "snap_found": True,
        "egm_offset_used": round(egm_to_bs77_offset, 3),
        "passport_bs77": h_passport,
        "target_egm": round(target_h, 3),
        "orig_lon": lon, "orig_lat": lat,
        "snap_lon": float(bl), "snap_lat": float(bla),
        "snap_distance_m": round(float(cand_dist[best]), 2),
        "dem_at_snap_egm": round(float(tile[best_row, best_col]), 3),
        "delta_h_m": round(float(cand_dh[best]), 3),
        "window_std_m": round(float(cand_std[best]), 3),
        "icesat2_support": bool(atl_support[best]),
        "icesat2_dz_med_m": (round(float(atl_dz_med[best]), 3)
                              if not np.isnan(atl_dz_med[best]) else None),
        "n_candidates": n_cands,
        "metric": round(float(metric[best]), 3),
        "tile": tile, "tile_origin": (c0, r0, src),
        "orig_xy": (cx, cy), "best_xy": (bx, by),
    }


# =============================================================================
#                      ПОИСК ОПТИМАЛЬНОГО offset_egm→bs77
# =============================================================================

def optimize_offset(src, atl_xy, atl_dz, atl_tree) -> dict:
    """
    Перебор OFFSET_GRID. Для каждого значения снапим только калибровочные
    точки и считаем суммарную метрику качества.
    Лучший offset — где сумма |Δh| + 2·std минимальна.
    """
    cal_gcps = [g for g in GCPS if g[4] == "calibration"]
    print(f"\nОптимизация EGM2008→БС-77 offset на {len(cal_gcps)} калибровочных точках:")
    print(f"  сетка: {OFFSET_GRID[0]:.2f}..{OFFSET_GRID[-1]:.2f} м, "
          f"шаг {OFFSET_GRID[1]-OFFSET_GRID[0]:.2f}")
    print()
    print(f"  {'offset':>8}  {'∑|Δh|':>8}  {'∑std':>8}  {'metric':>8}  notes")

    best_offset = None
    best_metric = np.inf
    history = []

    for offset in OFFSET_GRID:
        sum_dh = 0.0
        sum_std = 0.0
        n_found = 0
        for name, lon, lat, h, _ in cal_gcps:
            r = find_snap(name, lon, lat, h, float(offset), src,
                          atl_xy, atl_dz, atl_tree, verbose=False)
            if r["snap_found"]:
                sum_dh += r["delta_h_m"]
                sum_std += r["window_std_m"]
                n_found += 1

        if n_found < len(cal_gcps):
            note = f"!!! только {n_found}/{len(cal_gcps)} нашлись"
            metric = np.inf
        else:
            metric = sum_dh + 2 * sum_std
            note = ""

        history.append({
            "offset": round(float(offset), 3),
            "sum_dh": round(sum_dh, 3),
            "sum_std": round(sum_std, 3),
            "metric": round(float(metric), 3) if np.isfinite(metric) else None,
            "n_found": n_found,
        })
        marker = "  ←" if metric < best_metric else ""
        print(f"  {offset:>8.2f}  {sum_dh:>8.3f}  {sum_std:>8.3f}  "
              f"{metric:>8.3f}  {note}{marker}")
        if metric < best_metric:
            best_metric = metric
            best_offset = float(offset)

    print(f"\n  ★ Оптимальный offset = {best_offset:+.3f} м "
          f"(metric={best_metric:.3f})")
    return {"best_offset": best_offset, "history": history}


# =============================================================================
#                                   MAIN
# =============================================================================

def main() -> None:
    print(f"Читаю ATL08 точки...")
    atl08 = gpd.read_file(ATL08_GPKG, layer="atl08_with_dem")
    atl_xy = atl08[["x_m", "y_m"]].values
    atl_dz = atl08["dZ"].values
    atl_tree = cKDTree(atl_xy)
    print(f"  загружено: {len(atl08):,}")

    src = rasterio.open(DEM_EGM)
    print(f"DEM CRS: {src.crs}, size: {src.width}×{src.height}")

    # === 1. Водная кросс-калибровка ===
    print(f"\n{'=' * 70}")
    print(f"  ВОДНАЯ КРОСС-КАЛИБРОВКА")
    print(f"{'=' * 70}")
    water_levels = estimate_water_levels(src)
    for name, wl in water_levels.items():
        if wl is None:
            print(f"  {name}: нет данных")
        else:
            print(f"  {name}: медиана EGM = {wl['median_egm']:+.3f} м, "
                  f"std={wl['std_egm']:.3f} (n={wl['n_pixels']})")
    water_offsets = water_calibration_offset(water_levels)
    print(f"\n  Из воды получаем оценки сдвига EGM2008→БС-77:")
    for src_name, info in water_offsets.items():
        print(f"    {src_name}: water_BS77={info['water_bs77_assumed']:+.2f} − "
              f"water_EGM={info['water_egm']:+.2f} = "
              f"{info['offset_egm_to_bs77']:+.3f} м")

    # === 2. Optimize offset через калибровочные точки ===
    print(f"\n{'=' * 70}")
    print(f"  ОПТИМИЗАЦИЯ OFFSET ЧЕРЕЗ КАЛИБРОВОЧНЫЕ GCP")
    print(f"{'=' * 70}")
    opt = optimize_offset(src, atl_xy, atl_dz, atl_tree)
    optimal_offset = opt["best_offset"]

    # === 3. Сравниваем источники ===
    water_offsets_vals = [v["offset_egm_to_bs77"] for v in water_offsets.values()]
    print(f"\n{'=' * 70}")
    print(f"  СРАВНЕНИЕ ИСТОЧНИКОВ OFFSET")
    print(f"{'=' * 70}")
    print(f"  Из паспортов KZS-01/02:  {optimal_offset:+.3f} м")
    for src_name, info in water_offsets.items():
        print(f"  Из воды ({src_name}): {info['offset_egm_to_bs77']:+.3f} м")
    if water_offsets_vals:
        diff = optimal_offset - np.mean(water_offsets_vals)
        print(f"  Δ паспорт − вода:        {diff:+.3f} м")
        if abs(diff) < 0.3:
            print(f"  ✓ ИСТОЧНИКИ СОГЛАСУЮТСЯ — offset надёжен")
        elif abs(diff) < 0.6:
            print(f"  ⚠ источники расходятся, но в пределах допустимого")
        else:
            print(f"  ✗ ИСТОЧНИКИ СИЛЬНО РАСХОДЯТСЯ — нужна ручная проверка")

    # === 4. Финальный snap всех точек с оптимальным offset ===
    print(f"\n{'=' * 70}")
    print(f"  ФИНАЛЬНЫЙ SNAP ВСЕХ ТОЧЕК (offset = {optimal_offset:+.3f} м)")
    print(f"{'=' * 70}")
    results = []
    for name, lon, lat, h, role in GCPS:
        r = find_snap(name, lon, lat, h, optimal_offset, src,
                      atl_xy, atl_dz, atl_tree)
        r["role"] = role
        results.append(r)
        print(f"\n  [{role.upper()}] {name}:")
        if r["snap_found"]:
            print(f"     coord: lat={r['snap_lat']:.6f}, lon={r['snap_lon']:.6f} "
                  f"(Δ={r['snap_distance_m']:.1f} м)")
            print(f"     DEM:   {r['dem_at_snap_egm']:+.3f} → ожидали "
                  f"{r['target_egm']:+.3f}, |Δ|={r['delta_h_m']:.3f}")
            print(f"     std:   {r['window_std_m']:.3f} м, "
                  f"ICESat-2: {'✓' if r['icesat2_support'] else '✗'}")
            if r["icesat2_dz_med_m"] is not None:
                print(f"     ICESat-2 dZ: {r['icesat2_dz_med_m']:+.2f} м")
        else:
            print(f"     ✗ snap не найден")

    src.close()

    # === 5. Визуализация ===
    fig, axes = plt.subplots(2, 2, figsize=(13, 13))
    for ax, rec in zip(axes.flatten(), results):
        tile = rec["tile"]
        c0, r0, src_ref = rec["tile_origin"]
        if not np.isnan(tile).all():
            vmin = float(np.nanpercentile(tile, 2))
            vmax = float(np.nanpercentile(tile, 98))
        else:
            vmin, vmax = 0, 5
        im = ax.imshow(tile, cmap="terrain", vmin=vmin, vmax=vmax)

        cx, cy = rec["orig_xy"]
        pcol = (cx - src_ref.transform.c) / src_ref.transform.a - c0 - 0.5
        prow = (cy - src_ref.transform.f) / src_ref.transform.e - r0 - 0.5
        ax.plot(pcol, prow, marker="*", markersize=22, color="red",
                markeredgecolor="white", markeredgewidth=2,
                label="исходная (Yandex)")

        if rec.get("snap_found"):
            bx, by = rec["best_xy"]
            scol = (bx - src_ref.transform.c) / src_ref.transform.a - c0 - 0.5
            srow = (by - src_ref.transform.f) / src_ref.transform.e - r0 - 0.5
            color = "lime" if rec["role"] == "calibration" else "yellow"
            ax.plot(scol, srow, marker="o", markersize=18, color=color,
                    markeredgecolor="darkgreen", markeredgewidth=3,
                    label=f"snap [{rec['role']}] Δ={rec['snap_distance_m']:.0f}м")
            title = (f"{rec['name']} [{rec['role']}]\n"
                     f"target {rec['target_egm']:+.2f} → "
                     f"snap DEM={rec['dem_at_snap_egm']:+.2f}, "
                     f"|Δ|={rec['delta_h_m']:.2f}, std={rec['window_std_m']:.2f}\n"
                     f"ICESat-2: {'✓' if rec['icesat2_support'] else '✗'}")
        else:
            title = f"{rec['name']} [{rec['role']}]\n✗ snap не найден"
        ax.set_title(title, fontsize=10)
        ax.legend(loc="upper right", fontsize=8)
        plt.colorbar(im, ax=ax, label="м EGM2008", shrink=0.8)

    fig.suptitle(
        f"Snap v2 + water calibration: offset EGM→БС-77 = "
        f"{optimal_offset:+.3f} м",
        fontsize=11)
    fig.tight_layout()
    fig.savefig(OUT_PNG, dpi=130, bbox_inches="tight")
    plt.close(fig)
    print(f"\n✓ Карта снапов: {OUT_PNG}")

    # === 6. JSON отчёт ===
    serializable = []
    for r in results:
        out = {k: v for k, v in r.items()
               if k not in ("tile", "tile_origin", "orig_xy", "best_xy")}
        serializable.append(out)

    OUT_JSON.write_text(json.dumps({
        "optimal_offset_egm_to_bs77_m": optimal_offset,
        "water_calibration": {
            "levels": water_levels,
            "derived_offsets": water_offsets,
        },
        "offset_optimization": opt,
        "snap_results": serializable,
    }, indent=2, ensure_ascii=False))
    print(f"✓ JSON: {OUT_JSON}")

    # === 7. Готовая конфигурация ===
    print(f"\n{'=' * 70}")
    print(f"  ВЕРНУТЬ В spb_dem_build_v2.py")
    print(f"{'=' * 70}")
    print(f"\n  EGM2008→БС-77 offset = {optimal_offset:+.3f} м")
    print(f"\n  GCPS = [")
    for r in results:
        if r.get("snap_found") and r["role"] == "calibration":
            print(f"      GCP(\"{r['name']}\",")
            print(f"          lon={r['snap_lon']:.6f}, lat={r['snap_lat']:.6f}, "
                  f"h_bs77={r['passport_bs77']},")
            print(f"          source=\"snapped_v2 [calibration]\"),")
    print(f"  ]")
    print(f"\n  # validation-точки используем для оценки точности после калибровки")


if __name__ == "__main__":
    main()

Читаю ATL08 точки...
  загружено: 43,334
DEM CRS: EPSG:32636, size: 52224×33895

  ВОДНАЯ КРОСС-КАЛИБРОВКА
  KZS_finzaliv: медиана EGM = +0.361 м, std=0.716 (n=14323611)
  Neva_at_Gorny: медиана EGM = +0.862 м, std=0.489 (n=6762)

  Из воды получаем оценки сдвига EGM2008→БС-77:
    from_KZS_finzaliv: water_BS77=-0.10 − water_EGM=+0.36 = -0.461 м
    from_Neva_at_Gorny: water_BS77=-0.05 − water_EGM=+0.86 = -0.912 м

  ОПТИМИЗАЦИЯ OFFSET ЧЕРЕЗ КАЛИБРОВОЧНЫЕ GCP

Оптимизация EGM2008→БС-77 offset на 2 калибровочных точках:
  сетка: 0.80..2.20 м, шаг 0.10

    offset     ∑|Δh|      ∑std    metric  notes
      0.80     1.255     0.072     1.399    ←
      0.90     1.055     0.072     1.199    ←
      1.00     0.855     0.072     0.999    ←
      1.10     0.655     0.072     0.799    ←
      1.20     0.455     0.072     0.599    ←
      1.30     0.255     0.072     0.399    ←
      1.40     0.067     0.072     0.211    ←
      1.50     0.014     0.105     0.224  
      1.60     0.027     0.07

In [7]:
#!/usr/bin/env python3
"""
Финальная калибровка EGM2008 → БС-77 (трёхсторонняя проверка):

  A. ВОДА: offset = water_BS77 (по уровнемерам) − water_EGM (по DEM)
     По 3 постам (ГИ, С-1, С-2) с calm-фильтром.

  B. ПАСПОРТА с поправкой на возвышение объекта:
     паспорт даёт высоту "пятки мачты" / "репер на парапете" — точечных
     объектов, не разрешаемых ArcticDEM 2 м. Учитываем превышение:
       - метеомачта на КЗС: пятка примерно +1.7 м над служебной площадкой
       - репер II класса на парапете набережной: +1.1 м над мостовой
     ожидаемая_DEM_BS77 = паспорт − превышение_объекта

  C. ГРЕБЕНЬ ДАМБЫ (sanity check): по проекту КЗС +8.0 БС-77
     должен быть виден в DEM как локальный максимум вдоль трассы дамбы.

Запуск: python final_calibrate.py
"""

from __future__ import annotations

import json
from pathlib import Path

import numpy as np
import pandas as pd
import rasterio
from rasterio.windows import Window
from pyproj import Transformer


WORK_DIR = Path("./spb_dem")
DEM_EGM = WORK_DIR / "dem_egm2008.tif"
DEM_BS77_OUT = WORK_DIR / "dem_bs77.tif"
LEVELS_CSV = "/kaggle/input/datasets/ivanesin/gi-level/.csv"
OUT_JSON = WORK_DIR / "final_calibration.json"


# --- ПОСТЫ УРОВНЕМЕРОВ -------------------------------------------------------
GAUGES = {
    "GI_Gorny": {
        "csv_col": "ГИ",
        "water_bbox": (30.26, 59.925, 30.30, 59.94),
        "water_label": "Нева у Горного института",
    },
    "S1_Finzaliv_south": {
        "csv_col": "С1 ФЗ",
        "water_bbox": (29.65, 59.97, 29.74, 60.00),
        "water_label": "Финский залив у С-1 (юг)",
    },
    "S2_Finzaliv_north": {
        "csv_col": "С2 ФЗ",
        "water_bbox": (29.78, 60.01, 29.89, 60.04),
        "water_label": "Финский залив у С-2 (север)",
    },
}

# --- ПАСПОРТНЫЕ ТОЧКИ С УЧЁТОМ ПРИВЯЗКИ --------------------------------------
# obj_height_m = превышение точки паспорта над окружающей поверхностью DEM.
# Берём оценки осторожно, с реалистичной погрешностью ±0.3 м.
GCPS = [
    {"name": "KZS_01_V1",  "lon": 29.694727, "lat": 59.994069,
     "h_passport_bs77": 4.347, "object": "пятка мачты", "obj_height_m": 1.7,
     "snap_dem_egm": 2.953},
    {"name": "KZS_02_S1",  "lon": 29.696250, "lat": 59.987439,
     "h_passport_bs77": 3.367, "object": "пятка мачты", "obj_height_m": 1.7,
     "snap_dem_egm": 1.906},
    {"name": "KZS_03_S2_north", "lon": 29.835819, "lat": 60.028063,
     "h_passport_bs77": 3.770, "object": "пятка мачты (snap уехал)",
     "obj_height_m": 1.7, "snap_dem_egm": 1.667},
    {"name": "Gorny_embankment", "lon": 30.270442, "lat": 59.929320,
     "h_passport_bs77": 2.846, "object": "репер на парапете",
     "obj_height_m": 1.1, "snap_dem_egm": 2.348},
]

# --- ПРОВЕРКА ГРЕБНЯ ДАМБЫ --------------------------------------------------
# Проектная отметка гребня основной дамбы КЗС: +8.0 м БС-77 (публичные данные)
# Возьмём пиксели вдоль трассы и посмотрим P95 — это должен быть гребень.
DAM_TRANSECT = {
    "label": "Гребень основной дамбы КЗС (между С-1 и С-2)",
    "bbox": (29.69, 59.95, 29.85, 60.03),  # широкая полоса вдоль КЗС
    "expected_crest_bs77_m": 8.0,
    "tolerance_m": 1.0,
}

# --- ПАРАМЕТРЫ --------------------------------------------------------------
CALM_THRESHOLD_CM = 30
WATER_VALUE_RANGE_EGM = (-3, 1)
WATER_MIN_PIXELS = 1000


# =============================================================================

def load_levels() -> pd.DataFrame:
    df = pd.read_csv(LEVELS_CSV)
    df.iloc[:, 0] = pd.to_datetime(df.iloc[:, 0])
    df = df.rename(columns={df.columns[0]: "datetime"})
    for col in ["ГИ", "С1 ФЗ", "С2 ФЗ"]:
        if col in df.columns:
            df[col] = df[col] / 100.0
    return df


def gauge_summary(levels: pd.DataFrame, col: str) -> dict:
    s = levels[col].dropna()
    s_calm = s[s.abs() <= CALM_THRESHOLD_CM / 100]
    return {
        "n_total": int(len(s)),
        "median_m_all": round(float(s.median()), 3),
        "mean_m_all": round(float(s.mean()), 3),
        "median_m_calm": round(float(s_calm.median()), 3),
        "mean_m_calm": round(float(s_calm.mean()), 3),
    }


def water_dem_summary(src, bbox: tuple) -> dict:
    tr = Transformer.from_crs(4326, src.crs, always_xy=True)
    lo1, la1, lo2, la2 = bbox
    x1, y1 = tr.transform(lo1, la1); x2, y2 = tr.transform(lo2, la2)
    col1 = (min(x1, x2) - src.transform.c) / src.transform.a
    col2 = (max(x1, x2) - src.transform.c) / src.transform.a
    row1 = (max(y1, y2) - src.transform.f) / src.transform.e
    row2 = (min(y1, y2) - src.transform.f) / src.transform.e
    win = Window(int(min(col1, col2)), int(min(row1, row2)),
                 int(abs(col2-col1)), int(abs(row2-row1)))
    tile = src.read(1, window=win, boundless=True,
                    fill_value=np.nan).astype(float)
    if src.nodata is not None:
        tile[tile == src.nodata] = np.nan
    lo, hi = WATER_VALUE_RANGE_EGM
    flat = tile[(tile >= lo) & (tile <= hi)]
    if flat.size < WATER_MIN_PIXELS:
        return {"n_water_px": int(flat.size), "insufficient": True}
    return {
        "n_water_px": int(flat.size),
        "median_egm_m": round(float(np.median(flat)), 3),
        "p5_egm_m": round(float(np.quantile(flat, 0.05)), 3),
        "p95_egm_m": round(float(np.quantile(flat, 0.95)), 3),
    }


def dam_crest_check(src, dam_info: dict, offset: float) -> dict:
    """P95 высот в bbox дамбы должен быть около +8 м БС-77."""
    tr = Transformer.from_crs(4326, src.crs, always_xy=True)
    lo1, la1, lo2, la2 = dam_info["bbox"]
    x1, y1 = tr.transform(lo1, la1); x2, y2 = tr.transform(lo2, la2)
    col1 = (min(x1, x2) - src.transform.c) / src.transform.a
    col2 = (max(x1, x2) - src.transform.c) / src.transform.a
    row1 = (max(y1, y2) - src.transform.f) / src.transform.e
    row2 = (min(y1, y2) - src.transform.f) / src.transform.e
    win = Window(int(min(col1, col2)), int(min(row1, row2)),
                 int(abs(col2-col1)), int(abs(row2-row1)))
    tile = src.read(1, window=win, boundless=True,
                    fill_value=np.nan).astype(float)
    if src.nodata is not None:
        tile[tile == src.nodata] = np.nan
    valid = tile[~np.isnan(tile)]
    # Применяем offset → BS-77
    valid_bs77 = valid + offset
    p95 = float(np.quantile(valid_bs77, 0.95))
    p99 = float(np.quantile(valid_bs77, 0.99))
    pmax = float(valid_bs77.max())
    expected = dam_info["expected_crest_bs77_m"]
    tol = dam_info["tolerance_m"]
    return {
        "label": dam_info["label"],
        "expected_crest_bs77": expected,
        "tolerance": tol,
        "p95_bs77_m": round(p95, 2),
        "p99_bs77_m": round(p99, 2),
        "max_bs77_m": round(pmax, 2),
        "p95_within_tolerance": bool(abs(p95 - expected) <= tol),
        "interpretation": (
            f"гребень дамбы виден на ≈{p99:.1f} м БС-77, ожидаем "
            f"{expected}±{tol}; "
            f"{'OK' if abs(p99-expected) <= tol else 'отклонение'}"
        ),
    }


def sample_dem_at(src, lon: float, lat: float) -> float | None:
    """Билинейная выборка DEM в (lon, lat)."""
    tr = Transformer.from_crs(4326, src.crs, always_xy=True)
    x, y = tr.transform(lon, lat)
    col = (x - src.transform.c) / src.transform.a
    row = (y - src.transform.f) / src.transform.e
    c0, r0 = int(np.floor(col)), int(np.floor(row))
    dx, dy = col - c0, row - r0
    if not (0 <= r0 < src.height - 1 and 0 <= c0 < src.width - 1):
        return None
    win = Window(c0, r0, 2, 2)
    tile = src.read(1, window=win).astype(float)
    if src.nodata is not None:
        tile[tile == src.nodata] = np.nan
    if np.any(np.isnan(tile)):
        return None
    return float(
        (1-dx)*(1-dy)*tile[0,0] + dx*(1-dy)*tile[0,1]
        + (1-dx)*dy*tile[1,0] + dx*dy*tile[1,1])


def apply_offset_to_dem(dem_in, dem_out, offset: float) -> None:
    """Тайлово прибавляем константу."""
    with rasterio.open(dem_in) as src:
        profile = src.profile.copy()
        profile.update(dtype="float32", compress="LZW", predictor=3,
                       BIGTIFF="IF_SAFER")
        with rasterio.open(dem_out, "w", **profile) as dst:
            for _, window in src.block_windows(1):
                data = src.read(1, window=window).astype(np.float32)
                out = data + np.float32(offset)
                if src.nodata is not None:
                    out[data == src.nodata] = src.nodata
                dst.write(out, 1, window=window)


def main() -> None:
    print("=" * 72)
    print("  ФИНАЛЬНАЯ КАЛИБРОВКА: вода + паспорта + гребень дамбы")
    print("=" * 72)

    print(f"\nЗагружаю уровнемеры...")
    levels = load_levels()
    print(f"  записей: {len(levels):,}, период "
          f"{levels['datetime'].min()} → {levels['datetime'].max()}")

    src = rasterio.open(DEM_EGM)
    print(f"DEM: {DEM_EGM}, {src.width}×{src.height}")

    # === A. ВОДНАЯ КАЛИБРОВКА ================================================
    print(f"\n{'─' * 72}")
    print("  A. ВОДА — основная калибровка")
    print(f"{'─' * 72}")
    print(f"  {'Пост':<25} {'BS77':>10} {'BS77 calm':>12} {'DEM_egm':>10} "
          f"{'offset':>10}")
    water_results = {}
    for name, info in GAUGES.items():
        gs = gauge_summary(levels, info["csv_col"])
        ws = water_dem_summary(src, info["water_bbox"])
        if ws.get("insufficient"):
            print(f"  {name:<25}  ! недостаточно водных пикселей")
            continue
        water_dem = ws["median_egm_m"]
        offset_calm = round(gs["median_m_calm"] - water_dem, 3)
        offset_all = round(gs["median_m_all"] - water_dem, 3)
        water_results[name] = {
            "gauge": gs, "water_dem": ws,
            "offset_calm": offset_calm, "offset_all": offset_all,
            "label": info["water_label"],
        }
        print(f"  {name:<25} {gs['median_m_all']:>+10.3f} "
              f"{gs['median_m_calm']:>+12.3f} {water_dem:>+10.3f} "
              f"{offset_calm:>+10.3f}")

    # Финский залив — золотой эталон
    fz_offsets = [v["offset_calm"] for k, v in water_results.items()
                  if k.startswith("S")]
    if len(fz_offsets) >= 2:
        recommended_offset = round(float(np.median(fz_offsets)), 3)
        fz_std = float(np.std(fz_offsets))
        print(f"\n  Финский залив (С-1 vs С-2): offset = {fz_offsets[0]:+.3f} / "
              f"{fz_offsets[1]:+.3f}, std={fz_std:.3f} м")
        print(f"  ★ ОСНОВНОЙ OFFSET = {recommended_offset:+.3f} м "
              f"(медиана по С-1 и С-2)")
    else:
        print("\n  ⚠ Финский залив недоступен, используем медиану всех постов")
        all_off = [v["offset_calm"] for v in water_results.values()]
        recommended_offset = round(float(np.median(all_off)), 3)
        fz_std = float(np.std(all_off))
        print(f"  ★ OFFSET = {recommended_offset:+.3f} м (медиана)")

    gi_offset = water_results.get("GI_Gorny", {}).get("offset_calm")
    if gi_offset is not None and len(fz_offsets) >= 1:
        diff_gi_fz = round(gi_offset - np.median(fz_offsets), 3)
        print(f"\n  ГИ vs Финский: разница в offset = {diff_gi_fz:+.3f} м")
        print(f"     (Нева в DEM выше залива на эту величину — артефакт fill,")
        print(f"      или реальный градиент. На калибровку города не влияет.)")

    # === B. ПАСПОРТА С ПОПРАВКОЙ НА ВОЗВЫШЕНИЕ ОБЪЕКТА =======================
    print(f"\n{'─' * 72}")
    print("  B. ПАСПОРТА — валидация с учётом «пятки мачты» / «репер парапета»")
    print(f"{'─' * 72}")
    print(f"  Применяем offset = {recommended_offset:+.3f} к DEM_EGM, "
          "сравниваем с паспортом−возвышение_объекта")
    print()
    print(f"  {'Точка':<22} {'паспорт':>9} {'-h_obj':>8} {'ожид_BS77':>11} "
          f"{'DEM_BS77':>10} {'Δ':>8}")
    passport_results = []
    for g in GCPS:
        # фактический DEM в БС-77 на snap-позиции
        dem_bs77 = round(g["snap_dem_egm"] + recommended_offset, 3)
        expected_bs77 = round(g["h_passport_bs77"] - g["obj_height_m"], 3)
        delta = round(dem_bs77 - expected_bs77, 3)
        passport_results.append({
            "name": g["name"],
            "h_passport_bs77": g["h_passport_bs77"],
            "object": g["object"],
            "obj_height_m": g["obj_height_m"],
            "expected_dem_bs77": expected_bs77,
            "actual_dem_bs77": dem_bs77,
            "delta_m": delta,
        })
        flag = "✓" if abs(delta) <= 0.5 else ("~" if abs(delta) <= 1.0 else "✗")
        print(f"  {g['name']:<22} {g['h_passport_bs77']:>+9.2f} "
              f"{-g['obj_height_m']:>+8.2f} {expected_bs77:>+11.3f} "
              f"{dem_bs77:>+10.3f} {delta:>+8.3f} {flag}")

    deltas = np.array([r["delta_m"] for r in passport_results])
    rmse = float(np.sqrt(np.mean(deltas**2)))
    bias = float(np.mean(deltas))
    print(f"\n  По 4 паспортам с object-correction:  bias={bias:+.2f} м, "
          f"RMSE={rmse:.2f} м")
    if rmse < 0.5:
        print(f"  ✓ паспорта согласуются с DEM в пределах ожидаемой "
              f"неопределённости (~0.3-0.5 м)")
    elif rmse < 1.0:
        print(f"  ~ паспорта в пределах увеличенной неопределённости — "
              f"возможно obj_height_m надо подкрутить")
    else:
        print(f"  ✗ паспорта расходятся — пересмотреть obj_height_m или snap")

    # === C. ГРЕБЕНЬ ДАМБЫ ====================================================
    print(f"\n{'─' * 72}")
    print("  C. ГРЕБЕНЬ ДАМБЫ — независимая sanity-проверка")
    print(f"{'─' * 72}")
    dam_check = dam_crest_check(src, DAM_TRANSECT, recommended_offset)
    print(f"  {dam_check['label']}")
    print(f"  Ожидаемая отметка гребня: +{dam_check['expected_crest_bs77']} "
          f"± {dam_check['tolerance']} м БС-77")
    print(f"  В DEM (после offset {recommended_offset:+.3f}):")
    print(f"     P95: {dam_check['p95_bs77_m']:+.2f} м БС-77")
    print(f"     P99: {dam_check['p99_bs77_m']:+.2f} м БС-77")
    print(f"     max: {dam_check['max_bs77_m']:+.2f} м БС-77")
    print(f"  → {dam_check['interpretation']}")

    src.close()

    # === D. ПРИМЕНЯЕМ OFFSET К РАСТРУ И СОЗДАЁМ ФИНАЛЬНЫЙ DEM_BS77 ==========
    print(f"\n{'─' * 72}")
    print(f"  D. Применяем offset {recommended_offset:+.3f} → {DEM_BS77_OUT}")
    print(f"{'─' * 72}")
    apply_offset_to_dem(DEM_EGM, DEM_BS77_OUT, recommended_offset)
    print(f"  ✓ записан финальный DEM в БС-77")

    # === Сохраняем отчёт ===
    report = {
        "recommended_offset_egm_to_bs77_m": recommended_offset,
        "offset_source": "Финский залив (С-1 + С-2), медиана calm",
        "offset_uncertainty_m": round(fz_std, 3),
        "water_calibration": water_results,
        "passport_validation_with_object_correction": {
            "bias_m": round(bias, 3),
            "rmse_m": round(rmse, 3),
            "per_point": passport_results,
        },
        "dam_crest_sanity": dam_check,
        "params": {
            "calm_threshold_cm": CALM_THRESHOLD_CM,
            "water_value_range_egm": list(WATER_VALUE_RANGE_EGM),
        },
    }
    OUT_JSON.write_text(json.dumps(report, indent=2, ensure_ascii=False))
    print(f"\n✓ Отчёт: {OUT_JSON}")
    print(f"✓ Финальный DEM (БС-77): {DEM_BS77_OUT}")

    # Сводка как заголовок
    print(f"\n{'═' * 72}")
    print(f"  ИТОГ")
    print(f"{'═' * 72}")
    print(f"  offset EGM2008 → БС-77: {recommended_offset:+.3f} м "
          f"(±{fz_std:.3f} из 2 постов Финского)")
    print(f"  паспорта с обратной поправкой:  RMSE = {rmse:.2f} м, "
          f"bias = {bias:+.2f} м")
    print(f"  гребень дамбы (P99): {dam_check['p99_bs77_m']:+.2f} м БС-77 "
          f"(ожидаем ~{dam_check['expected_crest_bs77']:+.0f})")
    print(f"\n  Для финального пайплайна: offset_EGM_to_BS77 "
          f"= {recommended_offset}")


if __name__ == "__main__":
    main()

  ФИНАЛЬНАЯ КАЛИБРОВКА: вода + паспорта + гребень дамбы

Загружаю уровнемеры...
  записей: 84,762, период 2020-09-01 00:00:00 → 2023-02-01 00:00:00
DEM: spb_dem/dem_egm2008.tif, 52224×33895

────────────────────────────────────────────────────────────────────────
  A. ВОДА — основная калибровка
────────────────────────────────────────────────────────────────────────
  Пост                            BS77    BS77 calm    DEM_egm     offset
  GI_Gorny                      +0.240       +0.120     +0.715     -0.595
  S1_Finzaliv_south             +0.070       +0.030     +0.544     -0.514
  S2_Finzaliv_north             +0.060       +0.020     +0.305     -0.285

  Финский залив (С-1 vs С-2): offset = -0.514 / -0.285, std=0.115 м
  ★ ОСНОВНОЙ OFFSET = -0.399 м (медиана по С-1 и С-2)

  ГИ vs Финский: разница в offset = -0.196 м
     (Нева в DEM выше залива на эту величину — артефакт fill,
      или реальный градиент. На калибровку города не влияет.)

─────────────────────────────────────────

In [8]:
#!/usr/bin/env python3
"""
Калибровка через OSM-полигоны больших ровных объектов.

Часть 1 — внутренняя верификация (эллипсоид):
  Скачиваем большие плоские OSM-объекты (ВПП, апроны, большие парковки),
  для каждого считаем dZ = ATL08 − DEM в эллипсоидальных высотах внутри
  полигона. Если MAD внутри плоских объектов сильно меньше общегородского
  (0.75 м) — значит ICESat-2 коррегистрация работает на ровных местах
  идеально, и DEM в относительных высотах правильный.

Часть 2 — абсолютная привязка (эллипсоид → БС-77):
  Для аэродромов с известной опубликованной отметкой
  (Пулково ≈+24 м MSL, Ржевка ≈+17 м) считаем offset.
  Это слабая привязка (AIP даёт field elevation, не среднее по ВПП),
  но даёт независимую проверку.

Зависит от уже посчитанных артефактов:
    spb_dem/atl08_ground.gpkg     (ICESat-2 точки с DEM-сэмплом)
    spb_dem/dem_egm2008.tif       (DEM в EGM2008)
    spb_dem/dem_ellipsoidal.tif   (DEM до перевода в EGM)

Зависимости:
    pip install osmnx geopandas rasterio shapely
"""

from __future__ import annotations

import json
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.windows import from_bounds
from rasterio.features import geometry_mask
from shapely.geometry import box, Polygon
from pyproj import Transformer


WORK_DIR = Path("./spb_dem")
DEM_EGM = WORK_DIR / "dem_egm2008.tif"
ATL08_GPKG = WORK_DIR / "atl08_ground.gpkg"

OUT_GPKG = WORK_DIR / "osm_flat_polygons.gpkg"
OUT_JSON = WORK_DIR / "osm_calibration.json"

# AOI — то же что в основном пайплайне
AOI_BBOX = (29.5, 59.7, 30.7, 60.2)
WORK_EPSG = 32636

# --- Объекты для калибровки --------------------------------------------------
# OSM теги, которыми маркируются большие ровные поверхности в СПб
OSM_TAGS = {
    "runway":        {"aeroway": ["runway"]},
    "apron":         {"aeroway": ["apron", "taxiway"]},
    "parking_lot":   {"amenity": ["parking"]},
    "industrial_pad": {"landuse": ["industrial"]},
    "square":        {"place": ["square"]},
    "pitch":         {"leisure": ["pitch", "stadium"]},
}

# Минимальные размеры полигонов (м²) — отсеять мелочь
MIN_AREA_M2 = {
    "runway": 5000,         # 100×50 м минимум
    "apron": 5000,
    "parking_lot": 3000,    # большая парковка
    "industrial_pad": 10000,  # промплощадка ≥ 1 га
    "square": 2000,
    "pitch": 4000,          # стадион/большое поле
}

# --- Известные опубликованные отметки (AIP / public data) -------------------
# Источник + комментарий обязательны, чтобы было видно происхождение цифры.
KNOWN_ELEVATIONS_BS77 = [
    {
        "name": "Pulkovo_field",
        "tag_class": "runway",  # для какого класса искать пересечение
        "filter_lon_lat": (30.257, 59.799),  # центр ВПП 28L Пулково
        "search_radius_m": 1500,
        "elevation_bs77_m": 24.0,
        "tolerance_m": 3.0,    # field elevation — макс. по полю
        "source": "openAIP field elevation Пулково ULLI",
        "interpretation": "field elevation (макс. по полю), реальная "
                          "медиана ВПП может быть на 1-3 м ниже",
    },
    # Ржевка — закрытый аэродром, ВПП ещё цела.
    # Field elevation 56 ft ≈ 17.07 м MSL.
    # Но ВПП используется как стоянка автомобилей, поэтому DEM шумит.
    # {"name": "Rzhevka_runway", ...}
]


# =============================================================================
#                          1. ЗАГРУЗКА OSM-ПОЛИГОНОВ
# =============================================================================

def fetch_osm_flat_polygons() -> gpd.GeoDataFrame:
    """Скачиваем большие плоские объекты по тегам и фильтруем по площади."""
    import osmnx as ox
    ox.settings.timeout = 240
    ox.settings.requests_timeout = 240

    frames = []
    for class_name, tags in OSM_TAGS.items():
        try:
            print(f"  загружаю {class_name}: tags={tags}")
            g = ox.features_from_bbox(bbox=AOI_BBOX, tags=tags)
            if g.empty:
                print(f"    пусто")
                continue
            g = g[g.geometry.type.isin(["Polygon", "MultiPolygon"])]
            if g.empty:
                continue
            g = g.to_crs(WORK_EPSG)
            g["area_m2"] = g.geometry.area
            g = g[g["area_m2"] >= MIN_AREA_M2[class_name]].copy()
            g["class"] = class_name
            # Берём минимум полей чтобы не тащить весь мусор
            keep = ["class", "area_m2", "geometry"]
            extra = [c for c in ["name", "ref"] if c in g.columns]
            g = g[keep + extra].copy()
            print(f"    подходящих ≥{MIN_AREA_M2[class_name]} м²: {len(g)}")
            frames.append(g)
        except Exception as e:
            print(f"    ! {class_name}: {e}")

    if not frames:
        raise SystemExit("Не удалось скачать OSM полигоны")

    out = pd.concat(frames, ignore_index=True)
    out = gpd.GeoDataFrame(out, geometry="geometry", crs=WORK_EPSG)
    print(f"\n  Всего полигонов: {len(out)}")
    print(f"  По классам:")
    for cls in sorted(out["class"].unique()):
        sub = out[out["class"] == cls]
        total_ha = sub["area_m2"].sum() / 10000
        print(f"    {cls:<18} {len(sub):>5} полигонов, {total_ha:>7.1f} га")
    return out


# =============================================================================
#       2. dZ ATL08 - DEM ВНУТРИ ПОЛИГОНОВ (внутренняя верификация)
# =============================================================================

def dz_inside_polygons(polygons: gpd.GeoDataFrame,
                       atl08: gpd.GeoDataFrame) -> dict:
    """
    Для каждого класса OSM-объектов:
      берём ATL08-точки, попавшие внутрь полигонов этого класса,
      смотрим dZ = ATL08 - DEM (в эллипсоидальных высотах, уже посчитан).
    """
    if "dZ" not in atl08.columns:
        raise SystemExit("В atl08_ground.gpkg нет колонки dZ; "
                         "запустите spb_dem_build_v2.py до конца")

    # ATL08 уже в EPSG:WORK_EPSG (по x_m/y_m)
    atl_in_work = atl08.copy()
    if atl08.crs is None or atl08.crs.to_epsg() != WORK_EPSG:
        atl_in_work = atl08.to_crs(WORK_EPSG)

    print(f"\n  ATL08 точек всего: {len(atl_in_work):,}")
    overall_mad = float(np.median(
        np.abs(atl_in_work["dZ"] - np.median(atl_in_work["dZ"]))))
    print(f"  Общий MAD dZ:      {overall_mad:.3f} м "
          f"(median dZ = {atl_in_work['dZ'].median():+.3f} м)")

    results = {}
    print(f"\n  {'класс':<18} {'n_pts':>8} {'median dZ':>12} {'MAD':>10} "
          f"{'MAD/общий':>11}")
    for class_name in sorted(polygons["class"].unique()):
        sub_polys = polygons[polygons["class"] == class_name]
        # spatial join
        joined = gpd.sjoin(atl_in_work, sub_polys[["geometry"]],
                            how="inner", predicate="within")
        if len(joined) < 30:
            print(f"  {class_name:<18} {len(joined):>8}  ! мало точек, пропуск")
            results[class_name] = {"n_points": int(len(joined)),
                                   "skipped": True}
            continue

        med = float(np.median(joined["dZ"]))
        mad = float(np.median(np.abs(joined["dZ"] - med)))
        mad_norm = float(mad / overall_mad) if overall_mad > 0 else None

        results[class_name] = {
            "n_points": int(len(joined)),
            "median_dZ_m": round(med, 3),
            "mad_dZ_m": round(mad, 3),
            "mad_relative_to_global": round(mad_norm, 3) if mad_norm else None,
            "n_polygons": int(len(sub_polys)),
            "total_area_ha": round(float(sub_polys["area_m2"].sum() / 10000), 1),
        }
        print(f"  {class_name:<18} {len(joined):>8} {med:>+12.3f} "
              f"{mad:>10.3f} {mad_norm:>11.2f}")

    results["_overall"] = {
        "n_points": int(len(atl_in_work)),
        "median_dZ_m": round(float(atl_in_work["dZ"].median()), 3),
        "mad_dZ_m": round(overall_mad, 3),
    }
    return results


# =============================================================================
#         3. КРОСС-ПРОВЕРКА: DEM_EGM медиана внутри полигонов
# =============================================================================

def dem_egm_inside_polygons(polygons: gpd.GeoDataFrame) -> dict:
    """
    Для каждого класса считаем медиану и std DEM_EGM внутри полигонов.
    Это даёт картинку 'что DEM показывает в EGM2008 на каждом классе'.
    Полезно для интерпретации.
    """
    print(f"\n  Медианы DEM_EGM по классам OSM:")
    print(f"  {'класс':<18} {'n_polys':>8} {'med_egm':>10} {'mean_egm':>10} "
          f"{'std_inter':>10}")

    src = rasterio.open(DEM_EGM)
    polys_dem_crs = polygons.to_crs(src.crs)

    results = {}
    for class_name in sorted(polygons["class"].unique()):
        per_poly_med = []
        for geom in polys_dem_crs[polys_dem_crs["class"] == class_name].geometry:
            try:
                xmin, ymin, xmax, ymax = geom.bounds
                window = from_bounds(xmin, ymin, xmax, ymax, src.transform)
                tile = src.read(1, window=window).astype(float)
                if tile.size == 0:
                    continue
                if src.nodata is not None:
                    tile[tile == src.nodata] = np.nan
                # маска полигона внутри окна
                w_transform = rasterio.windows.transform(window, src.transform)
                mask = geometry_mask([geom], transform=w_transform,
                                      invert=True, out_shape=tile.shape)
                vals = tile[mask & ~np.isnan(tile)]
                if vals.size >= 50:   # хотя бы 50 пикселей
                    per_poly_med.append(float(np.median(vals)))
            except Exception:
                continue

        if not per_poly_med:
            results[class_name] = {"skipped": True}
            continue

        arr = np.array(per_poly_med)
        results[class_name] = {
            "n_polygons_with_data": int(len(arr)),
            "median_dem_egm_m": round(float(np.median(arr)), 3),
            "mean_dem_egm_m": round(float(np.mean(arr)), 3),
            "std_inter_polygons_m": round(float(np.std(arr)), 3),
            "p5_m": round(float(np.quantile(arr, 0.05)), 3),
            "p95_m": round(float(np.quantile(arr, 0.95)), 3),
        }
        print(f"  {class_name:<18} {len(arr):>8} "
              f"{np.median(arr):>+10.3f} {np.mean(arr):>+10.3f} "
              f"{np.std(arr):>10.3f}")

    src.close()
    return results


# =============================================================================
#           4. КАЛИБРОВКА ПО ОПУБЛИКОВАННЫМ ОТМЕТКАМ (AIP)
# =============================================================================

def calibrate_by_known_elevations(polygons: gpd.GeoDataFrame) -> list[dict]:
    """
    Для каждой записи в KNOWN_ELEVATIONS_BS77 находим ближайший подходящий
    OSM-полигон, смотрим медиану DEM_EGM в нём, считаем offset.
    """
    print(f"\n{'─' * 72}")
    print("  4. Калибровка по известным AIP-отметкам")
    print(f"{'─' * 72}")

    src = rasterio.open(DEM_EGM)
    tr = Transformer.from_crs(4326, src.crs, always_xy=True)

    results = []
    for entry in KNOWN_ELEVATIONS_BS77:
        cls = entry["tag_class"]
        sub = polygons[polygons["class"] == cls]
        if sub.empty:
            print(f"  {entry['name']}: класс '{cls}' пуст, пропуск")
            continue

        cx, cy = tr.transform(*entry["filter_lon_lat"])
        sub_xy = sub.copy()
        sub_xy["dist"] = np.hypot(
            sub_xy.geometry.centroid.x - cx,
            sub_xy.geometry.centroid.y - cy)
        sub_xy = sub_xy[sub_xy["dist"] <= entry["search_radius_m"]]
        if sub_xy.empty:
            print(f"  {entry['name']}: нет полигонов в радиусе "
                  f"{entry['search_radius_m']} м")
            continue

        # Берём самый большой (предположительно — это и есть ВПП)
        biggest = sub_xy.sort_values("area_m2", ascending=False).iloc[0]
        geom = biggest.geometry

        try:
            xmin, ymin, xmax, ymax = geom.bounds
            window = from_bounds(xmin, ymin, xmax, ymax, src.transform)
            tile = src.read(1, window=window).astype(float)
            if src.nodata is not None:
                tile[tile == src.nodata] = np.nan
            w_transform = rasterio.windows.transform(window, src.transform)
            mask = geometry_mask([geom], transform=w_transform,
                                  invert=True, out_shape=tile.shape)
            vals = tile[mask & ~np.isnan(tile)]
            if vals.size < 100:
                print(f"  {entry['name']}: мало валидных пикселей")
                continue

            med = float(np.median(vals))
            p95 = float(np.quantile(vals, 0.95))
            offset_med = round(entry["elevation_bs77_m"] - med, 3)
            offset_p95 = round(entry["elevation_bs77_m"] - p95, 3)

            print(f"\n  {entry['name']} (площадь {biggest['area_m2']/1e4:.1f} га):")
            print(f"    DEM_EGM медиана: {med:+.3f} м, P95: {p95:+.3f} м "
                  f"(n_px={vals.size:,})")
            print(f"    AIP отметка БС-77: {entry['elevation_bs77_m']:+.2f} ± "
                  f"{entry['tolerance_m']} м ({entry['source']})")
            print(f"    offset (медиана) = {offset_med:+.3f} м")
            print(f"    offset (P95)     = {offset_p95:+.3f} м "
                  f"← ближе к 'field elevation'")
            print(f"    интерпретация: {entry['interpretation']}")

            results.append({
                "name": entry["name"],
                "elevation_bs77_m": entry["elevation_bs77_m"],
                "tolerance_m": entry["tolerance_m"],
                "polygon_area_ha": round(biggest["area_m2"]/1e4, 2),
                "n_pixels": int(vals.size),
                "dem_egm_median": round(med, 3),
                "dem_egm_p95": round(p95, 3),
                "offset_median": offset_med,
                "offset_p95": offset_p95,
                "source": entry["source"],
            })
        except Exception as e:
            print(f"  ! {entry['name']}: {e}")

    src.close()
    return results


# =============================================================================
#                                ГЛАВНЫЙ
# =============================================================================

def main() -> None:
    if not ATL08_GPKG.exists():
        raise SystemExit(f"Нет {ATL08_GPKG} — сначала запустите spb_dem_build_v2.py")
    if not DEM_EGM.exists():
        raise SystemExit(f"Нет {DEM_EGM}")

    print("=" * 72)
    print("  OSM-калибровка")
    print("=" * 72)

    print(f"\n[1/4] Скачиваем OSM-полигоны")
    polys = fetch_osm_flat_polygons()
    polys.to_file(OUT_GPKG, driver="GPKG", layer="osm_flat")
    print(f"  → сохранено: {OUT_GPKG}")

    print(f"\n[2/4] dZ = ATL08 − DEM (эллипсоид) внутри полигонов")
    print("─" * 72)
    atl08 = gpd.read_file(ATL08_GPKG, layer="atl08_with_dem")
    dz_results = dz_inside_polygons(polys, atl08)

    print(f"\n[3/4] DEM_EGM медианы внутри полигонов (диагностика)")
    print("─" * 72)
    egm_results = dem_egm_inside_polygons(polys)

    print(f"\n[4/4] Калибровка по AIP отметкам")
    aip_results = calibrate_by_known_elevations(polys)

    # --- Интерпретация ------------------------------------------------------
    print(f"\n{'═' * 72}")
    print("  ИНТЕРПРЕТАЦИЯ")
    print(f"{'═' * 72}")

    overall_mad = dz_results["_overall"]["mad_dZ_m"]
    overall_med = dz_results["_overall"]["median_dZ_m"]

    # Лучшие по точности классы
    flat_classes = []
    for cls, r in dz_results.items():
        if cls.startswith("_") or r.get("skipped"):
            continue
        if r.get("mad_dZ_m") and r["mad_dZ_m"] < 0.5:
            flat_classes.append((cls, r))

    if flat_classes:
        flat_classes.sort(key=lambda x: x[1]["mad_dZ_m"])
        print(f"\n  Самые ровные классы (по MAD dZ):")
        for cls, r in flat_classes[:5]:
            improv = (1 - r["mad_dZ_m"] / overall_mad) * 100 if overall_mad else 0
            print(f"    {cls:<18} MAD={r['mad_dZ_m']:.3f} м, "
                  f"улучшение {improv:.0f}% vs общегородского {overall_mad:.2f}")

    if aip_results:
        offsets = [r["offset_median"] for r in aip_results]
        offsets_p95 = [r["offset_p95"] for r in aip_results]
        print(f"\n  AIP-калибровка (медиана и P95 по ВПП):")
        for r in aip_results:
            print(f"    {r['name']:<18} offset_med={r['offset_median']:+.2f}, "
                  f"offset_p95={r['offset_p95']:+.2f}")
        print(f"\n  Внимание: AIP даёт field elevation = максимум по полю.")
        print(f"  Если offset_p95 близок к нему — DEM правильно показывает "
              f"максимум.")
        print(f"  offset_median обычно ниже — это offset для среднего по ВПП.")

    print(f"\n  Что НЕ закрывает эта калибровка:")
    print(f"  - Абсолютная привязка эллипсоид → БС-77 остаётся "
          f"неопределённой в пределах ~1-2 м, потому что AIP даёт ")
    print(f"    'field elevation', а не среднюю отметку.")
    print(f"  - Для строгой привязки нужна одна геодезически точная ")
    print(f"    точка БС-77 на ровной поверхности (тротуар/площадь),")
    print(f"    обмеренная GNSS-приёмником на текущей высоте — без мачт")
    print(f"    и парапетов.")

    # --- Сохранить -----------------------------------------------------------
    out = {
        "atl08_dz_inside_polygons": dz_results,
        "dem_egm_inside_polygons": egm_results,
        "aip_calibration": aip_results,
        "config": {
            "aoi_bbox": list(AOI_BBOX),
            "min_area_m2": MIN_AREA_M2,
            "n_total_polygons": len(polys),
        },
    }
    OUT_JSON.write_text(json.dumps(out, indent=2, ensure_ascii=False))
    print(f"\n✓ Отчёт: {OUT_JSON}")
    print(f"✓ OSM полигоны: {OUT_GPKG}")


if __name__ == "__main__":
    main()

  OSM-калибровка

[1/4] Скачиваем OSM-полигоны
  загружаю runway: tags={'aeroway': ['runway']}
    подходящих ≥5000 м²: 1
  загружаю apron: tags={'aeroway': ['apron', 'taxiway']}
    подходящих ≥5000 м²: 20
  загружаю parking_lot: tags={'amenity': ['parking']}
    подходящих ≥3000 м²: 1180
  загружаю industrial_pad: tags={'landuse': ['industrial']}
    подходящих ≥10000 м²: 1820
  загружаю square: tags={'place': ['square']}
    подходящих ≥2000 м²: 113
  загружаю pitch: tags={'leisure': ['pitch', 'stadium']}
    подходящих ≥4000 м²: 267

  Всего полигонов: 3401
  По классам:
    apron                 20 полигонов,   150.4 га
    industrial_pad      1820 полигонов, 15793.7 га
    parking_lot         1180 полигонов,   898.5 га
    pitch                267 полигонов,   277.3 га
    runway                 1 полигонов,    14.8 га
    square               113 полигонов,   256.9 га
  → сохранено: spb_dem/osm_flat_polygons.gpkg

[2/4] dZ = ATL08 − DEM (эллипсоид) внутри полигонов
─────────────

In [9]:
#!/usr/bin/env python3
"""
Калибровка по расширенной сети из 15 паспортных точек:
  - 4 старых (Gorny + KZS-01/02/03)
  - 11 новых (различные объекты в районе СПб)

Что делаем:
  1. Для каждой точки — окно 21×21 пикс + ближайшие ATL08, флаги.
  2. Строим robust подгонку плоскости: BS-77(x,y) = a + b·x + c·y по
     всем точкам с iterative выбраковкой выбросов (RANSAC-style).
  3. Применяем найденную плоскость к dem_egm2008.tif → dem_bs77.tif.
  4. Финальная LOO-валидация: для каждой точки предсказание по остальным.

Запуск:
    python calibrate_15.py
"""

from __future__ import annotations

import json
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.windows import Window
import matplotlib.pyplot as plt
from pyproj import Transformer
from scipy.spatial import cKDTree
from scipy.ndimage import generic_filter


WORK_DIR = Path("./spb_dem")
DEM_EGM = WORK_DIR / "dem_egm2008.tif"
DEM_BS77_OUT = WORK_DIR / "dem_bs77.tif"
ATL08_GPKG = WORK_DIR / "atl08_ground.gpkg"

OUT_DIAG_PNG = WORK_DIR / "gcp15_diagnostic.png"
OUT_REPORT = WORK_DIR / "gcp15_calibration.json"

WORK_EPSG = 32636

# --- ВСЕ ТОЧКИ ---------------------------------------------------------------
# Old + new. Все в БС-77, координаты WGS84.
ALL_POINTS = [
    # Старые
    ("Gorny",     30.269963, 59.929175, 2.846, "old"),
    ("KZS_01",    29.694830, 59.994314, 4.347, "old"),
    ("KZS_02",    29.696649, 59.987700, 3.367, "old"),
    ("KZS_03",    29.836873, 60.027940, 3.770, "old"),
    # Новые — район КЗС
    ("P01",       29.842406, 60.027048, 3.00,  "new"),
    ("P02",       29.845407, 60.026729, 4.91,  "new"),
    ("P03",       29.845589, 60.026572, 4.87,  "new"),
    ("P04",       29.841349, 60.026912, 3.00,  "new"),
    ("P05",       29.698166, 59.992611, 2.65,  "new"),
    ("P06",       29.835426, 60.025722, 3.25,  "new"),
    ("P07",       29.696296, 59.989239, 3.00,  "new"),
    ("P08",       29.721013, 60.012491, 3.20,  "new"),
    # Новые — район Левашово/Парголово
    ("P09",       30.108990, 60.019754, 5.27,  "new"),
    ("P10",       30.110281, 60.019732, 4.77,  "new"),
    # Новая — юго-восток
    ("P11",       30.214238, 59.878634, 3.50,  "new"),
]

# --- Параметры диагностики --------------------------------------------------
WIN_PX = 210              # окно 42×42 м
NEIGHBOR_K = 30
NEIGHBOR_MAX_M = 1000

# --- Параметры robust fit ---------------------------------------------------
RANSAC_THRESHOLD_M = 1.0  # точка-выброс если abs(residual) > этого
RANSAC_ITERATIONS = 5     # итеративное удаление выбросов


# =============================================================================

def diagnose_point(name: str, lon: float, lat: float, h_bs77: float,
                   src: rasterio.DatasetReader,
                   atl_xy: np.ndarray, atl_dz: np.ndarray,
                   atl_tree: cKDTree) -> dict:
    """Окно DEM + ICESat-2 окрестность."""
    tr = Transformer.from_crs(4326, src.crs, always_xy=True)
    x, y = tr.transform(lon, lat)

    col = (x - src.transform.c) / src.transform.a
    row = (y - src.transform.f) / src.transform.e
    c0 = int(round(col)) - WIN_PX // 2
    r0 = int(round(row)) - WIN_PX // 2
    win = Window(c0, r0, WIN_PX, WIN_PX)
    tile = src.read(1, window=win, boundless=True,
                    fill_value=np.nan).astype(float)
    if src.nodata is not None:
        tile[tile == src.nodata] = np.nan
    valid = tile[~np.isnan(tile)]
    if not valid.size:
        return {"name": name, "x_m": x, "y_m": y, "h_bs77": h_bs77,
                "dem_egm_center": None, "dem_egm_median": None,
                "tile": tile, "tile_origin": (c0, r0)}

    center = float(tile[WIN_PX // 2, WIN_PX // 2])
    med = float(np.nanmedian(valid))
    std = float(np.nanstd(valid))
    minv = float(np.nanmin(valid))
    maxv = float(np.nanmax(valid))

    # Ближайшие ATL08
    dists, idx = atl_tree.query([x, y], k=NEIGHBOR_K)
    keep = dists <= NEIGHBOR_MAX_M
    atl_summary = None
    if keep.any():
        nearby_dz = atl_dz[idx[keep]]
        atl_summary = {
            "n": int(keep.sum()),
            "min_dist_m": float(dists[keep].min()),
            "dz_med": float(np.median(nearby_dz)),
            "dz_mad": float(np.median(np.abs(nearby_dz - np.median(nearby_dz)))),
        }

    flags = []
    if std > 1.5:
        flags.append(f"высокая дисперсия std={std:.2f}")
    if maxv - minv > 5:
        flags.append(f"перепад в окне {maxv-minv:.1f} м")
    if abs(med) < 0.3:
        flags.append("медиана ~0 м EGM, возможна вода")

    return {
        "name": name,
        "lon": lon, "lat": lat,
        "x_m": x, "y_m": y,
        "h_bs77": h_bs77,
        "dem_egm_center": round(center, 3),
        "dem_egm_median": round(med, 3),
        "dem_egm_std": round(std, 3),
        "dem_egm_min": round(minv, 3),
        "dem_egm_max": round(maxv, 3),
        "icesat2": atl_summary,
        "flags": flags,
        "tile": tile,
        "tile_origin": (c0, r0),
    }


# =============================================================================
#                  ROBUST PLANE FIT (iterative RANSAC-style)
# =============================================================================

def robust_plane_fit(points: list[dict]) -> dict:
    """
    Подгоняем BS77(x, y) = a + b·(x-x0) + c·(y-y0)
    к данным dem_egm_center → h_bs77.

    Итеративно: подгоняем плоскость, выбрасываем точки с residual > threshold,
    повторяем. Останавливаемся, когда выбросов больше нет.
    """
    valid = [p for p in points if p.get("dem_egm_center") is not None]
    if len(valid) < 4:
        raise SystemExit(f"Слишком мало точек с dem_egm_center: {len(valid)}")

    # Подгонка: ожидаем h_bs77 ≈ dem_egm + offset(x,y)
    # Регрессируем (h_bs77 - dem_egm) на (x, y) — это делает геоидно-подобный
    # тренд интерпретируемым.

    x = np.array([p["x_m"] for p in valid])
    y = np.array([p["y_m"] for p in valid])
    h_bs = np.array([p["h_bs77"] for p in valid])
    h_egm = np.array([p["dem_egm_center"] for p in valid])
    delta = h_bs - h_egm    # это "offset" в каждой точке

    x0, y0 = float(np.mean(x)), float(np.mean(y))

    print(f"\n  Robust подгонка плоскости offset(x,y) = a + b·(x-x0) + c·(y-y0)")
    print(f"  Стартовый набор: {len(valid)} точек")
    print(f"  {'iter':>4} {'kept':>5} {'a (offset)':>11} {'b (м/км)':>9} "
          f"{'c (м/км)':>9} {'RMS':>6} {'max_res':>9}")

    keep_mask = np.ones(len(valid), dtype=bool)
    history = []

    for it in range(RANSAC_ITERATIONS + 1):
        xs = x[keep_mask] - x0
        ys = y[keep_mask] - y0
        ds = delta[keep_mask]

        if len(ds) < 4:
            print(f"    < 4 точек, останавливаюсь")
            break

        # Least squares
        A = np.column_stack([np.ones_like(xs), xs, ys])
        coef, *_ = np.linalg.lstsq(A, ds, rcond=None)
        a, b, c = map(float, coef)
        pred = a + b * xs + c * ys
        resid = ds - pred
        rms = float(np.sqrt(np.mean(resid**2)))
        max_res = float(np.max(np.abs(resid)))
        b_per_km = b * 1000
        c_per_km = c * 1000

        print(f"  {it:>4} {keep_mask.sum():>5} {a:>+11.3f} {b_per_km:>+9.3f} "
              f"{c_per_km:>+9.3f} {rms:>6.3f} {max_res:>9.3f}")

        history.append({
            "iter": it,
            "kept": int(keep_mask.sum()),
            "a": round(a, 3),
            "b_per_km": round(b_per_km, 3),
            "c_per_km": round(c_per_km, 3),
            "rms": round(rms, 3),
            "max_res": round(max_res, 3),
        })

        # Считаем residuals на всём наборе (включая выброшенные)
        all_pred = a + b * (x - x0) + c * (y - y0)
        all_resid = delta - all_pred

        # Какие сейчас выбросы?
        new_mask = np.abs(all_resid) <= RANSAC_THRESHOLD_M
        if np.array_equal(new_mask, keep_mask) or it == RANSAC_ITERATIONS:
            break
        keep_mask = new_mask

    # Финальные данные
    final_pred = a + b * (x - x0) + c * (y - y0)
    final_resid = delta - final_pred

    per_point = []
    for i, p in enumerate(valid):
        per_point.append({
            "name": p["name"],
            "h_bs77": p["h_bs77"],
            "dem_egm": p["dem_egm_center"],
            "delta_observed": round(delta[i], 3),
            "delta_predicted": round(final_pred[i], 3),
            "residual": round(final_resid[i], 3),
            "kept": bool(keep_mask[i]),
        })

    return {
        "x0_m": x0, "y0_m": y0,
        "a_offset_at_x0y0": round(a, 3),
        "b_per_m": b, "c_per_m": c,
        "b_per_km": round(b_per_km, 3),
        "c_per_km": round(c_per_km, 3),
        "rms_m": round(rms, 3),
        "n_kept": int(keep_mask.sum()),
        "n_total": len(valid),
        "iterations": history,
        "per_point": per_point,
    }


# =============================================================================
#                    LOO-валидация: leave-one-out
# =============================================================================

def loo_validation(points: list[dict], kept_names: set) -> dict:
    """LOO только по точкам, оставшимся после RANSAC."""
    valid = [p for p in points if p["name"] in kept_names
             and p.get("dem_egm_center") is not None]
    if len(valid) < 4:
        return {"skipped": True}

    x = np.array([p["x_m"] for p in valid])
    y = np.array([p["y_m"] for p in valid])
    h_bs = np.array([p["h_bs77"] for p in valid])
    h_egm = np.array([p["dem_egm_center"] for p in valid])
    delta = h_bs - h_egm

    residuals = []
    for i in range(len(valid)):
        mask = np.arange(len(valid)) != i
        x0, y0 = float(np.mean(x[mask])), float(np.mean(y[mask]))
        A = np.column_stack([
            np.ones(mask.sum()), x[mask] - x0, y[mask] - y0])
        coef, *_ = np.linalg.lstsq(A, delta[mask], rcond=None)
        a, b, c = map(float, coef)
        pred = a + b * (x[i] - x0) + c * (y[i] - y0)
        residuals.append({
            "name": valid[i]["name"],
            "actual_delta": round(float(delta[i]), 3),
            "predicted_delta": round(pred, 3),
            "residual": round(float(delta[i] - pred), 3),
        })

    rms = float(np.sqrt(np.mean([r["residual"]**2 for r in residuals])))
    max_abs = float(max(abs(r["residual"]) for r in residuals))
    return {"rms_m": round(rms, 3), "max_abs_m": round(max_abs, 3),
            "residuals": residuals}


# =============================================================================
#               ПРИМЕНЕНИЕ ПЛОСКОСТИ К РАСТРУ → DEM_BS77
# =============================================================================

def apply_plane_to_dem(dem_egm_path: Path, dem_bs77_path: Path,
                       fit: dict) -> None:
    """dem_bs77(x,y) = dem_egm(x,y) + (a + b·(x-x0) + c·(y-y0))."""
    a = fit["a_offset_at_x0y0"]
    b = fit["b_per_m"]; c = fit["c_per_m"]
    x0, y0 = fit["x0_m"], fit["y0_m"]

    with rasterio.open(dem_egm_path) as src:
        profile = src.profile.copy()
        profile.update(dtype="float32", compress="LZW", predictor=3,
                       BIGTIFF="IF_SAFER")
        tr_to_work = Transformer.from_crs(src.crs, WORK_EPSG, always_xy=True)
        with rasterio.open(dem_bs77_path, "w", **profile) as dst:
            for _, window in src.block_windows(1):
                data = src.read(1, window=window).astype(np.float32)
                rows, cols = np.mgrid[
                    window.row_off:window.row_off + window.height,
                    window.col_off:window.col_off + window.width]
                xs, ys = rasterio.transform.xy(
                    src.transform, rows.ravel(), cols.ravel())
                xm, ym = tr_to_work.transform(np.asarray(xs), np.asarray(ys))
                shift = (a + b * (xm.reshape(data.shape) - x0)
                         + c * (ym.reshape(data.shape) - y0)).astype(np.float32)
                out = data + shift
                if src.nodata is not None:
                    out[data == src.nodata] = src.nodata
                dst.write(out, 1, window=window)


# =============================================================================
#                                    MAIN
# =============================================================================

def main() -> None:
    print("=" * 72)
    print("  Калибровка по 15 паспортным точкам (4 старых + 11 новых)")
    print("=" * 72)

    src = rasterio.open(DEM_EGM)
    print(f"DEM: {DEM_EGM}, {src.width}×{src.height}")

    print(f"\nЧитаю ATL08...")
    atl08 = gpd.read_file(ATL08_GPKG, layer="atl08_with_dem")
    atl_xy = atl08[["x_m", "y_m"]].values
    atl_dz = atl08["dZ"].values
    atl_tree = cKDTree(atl_xy)
    print(f"  {len(atl08):,} точек")

    # === ШАГ 1: диагностика всех точек ===
    print(f"\n{'─' * 72}")
    print("  ДИАГНОСТИКА ПО КАЖДОЙ ТОЧКЕ")
    print(f"{'─' * 72}")
    print(f"  {'name':<8} {'h_bs77':>8} {'DEM_egm':>9} {'std':>6} "
          f"{'∆max':>5} {'ATL_dz':>7} {'expected_offset':>15}")

    points = []
    for name, lon, lat, h_bs77, role in ALL_POINTS:
        rec = diagnose_point(name, lon, lat, h_bs77, src, atl_xy, atl_dz, atl_tree)
        rec["role"] = role
        points.append(rec)

        if rec["dem_egm_center"] is None:
            print(f"  {name:<8}  ! NoData")
            continue

        atl_str = (f"{rec['icesat2']['dz_med']:+.2f}"
                   if rec.get("icesat2") else "n/a")
        offset = h_bs77 - rec["dem_egm_center"]
        flag = ""
        if rec["flags"]:
            flag = " ⚠"
        print(f"  {name:<8} {h_bs77:>+8.2f} {rec['dem_egm_center']:>+9.3f} "
              f"{rec['dem_egm_std']:>6.2f} {rec['dem_egm_max']-rec['dem_egm_min']:>5.1f} "
              f"{atl_str:>7} {offset:>+15.3f}{flag}")

    src.close()

    # === ШАГ 2: robust plane fit ===
    print(f"\n{'─' * 72}")
    print("  ROBUST ПОДГОНКА ПЛОСКОСТИ (RANSAC-style)")
    print(f"{'─' * 72}")
    fit = robust_plane_fit(points)

    print(f"\n  Финал: оставлено {fit['n_kept']}/{fit['n_total']} точек, "
          f"RMS={fit['rms_m']} м")
    print(f"  offset в центре: {fit['a_offset_at_x0y0']:+.3f} м")
    print(f"  тилт: b={fit['b_per_km']:+.3f} м/км, c={fit['c_per_km']:+.3f} м/км")

    # Покажем индивидуальные residuals
    print(f"\n  {'name':<8} {'observed':>9} {'predicted':>10} {'residual':>10}  status")
    for pp in fit["per_point"]:
        status = "✓" if pp["kept"] else "✗ выброс"
        print(f"  {pp['name']:<8} {pp['delta_observed']:>+9.3f} "
              f"{pp['delta_predicted']:>+10.3f} {pp['residual']:>+10.3f}  {status}")

    # === ШАГ 3: LOO ===
    print(f"\n{'─' * 72}")
    print("  LEAVE-ONE-OUT валидация (только на сохранённых точках)")
    print(f"{'─' * 72}")
    kept_names = {p["name"] for p in fit["per_point"] if p["kept"]}
    loo = loo_validation(points, kept_names)
    if loo.get("skipped"):
        print("  ! пропущено (мало точек)")
    else:
        print(f"  LOO RMS:    {loo['rms_m']} м")
        print(f"  LOO max:    {loo['max_abs_m']} м")

    # === ШАГ 4: применяем к растру ===
    print(f"\n{'─' * 72}")
    print(f"  Применяем плоскость к DEM_EGM → {DEM_BS77_OUT}")
    print(f"{'─' * 72}")
    apply_plane_to_dem(DEM_EGM, DEM_BS77_OUT, fit)
    print(f"  ✓ создан DEM в БС-77")

    # === ШАГ 5: визуализация (4×4 = 16 субплотов, 15 точек + общая карта) ===
    n = len(points)
    cols = 4
    rows = (n + cols - 1) // cols + 1   # +1 для финальной карты residuals
    fig, axes = plt.subplots(rows, cols, figsize=(16, 4 * rows))
    axes = axes.flatten()

    for i, p in enumerate(points):
        ax = axes[i]
        if p.get("tile") is None or np.isnan(p["tile"]).all():
            ax.set_title(f"{p['name']}: NoData", fontsize=9)
            ax.axis("off")
            continue
        tile = p["tile"]
        vmin = float(np.nanpercentile(tile, 2))
        vmax = float(np.nanpercentile(tile, 98))
        im = ax.imshow(tile, cmap="terrain", vmin=vmin, vmax=vmax)
        ax.plot(WIN_PX // 2, WIN_PX // 2, marker="*", markersize=14,
                color="red", markeredgecolor="white", markeredgewidth=1.5)
        # Найти этот name в fit для статуса
        status = ""
        for pp in fit["per_point"]:
            if pp["name"] == p["name"]:
                status = "✓" if pp["kept"] else "✗"
                break
        title = (f"{p['name']} [{p['role']}] {status}\n"
                 f"BS77={p['h_bs77']:.2f}, DEM_egm={p['dem_egm_center']:.2f}\n"
                 f"std={p['dem_egm_std']:.2f}, ∆={p['dem_egm_max']-p['dem_egm_min']:.1f}")
        ax.set_title(title, fontsize=8)
        ax.set_xticks([]); ax.set_yticks([])

    # Скрыть лишние
    for i in range(n, len(axes) - cols):
        axes[i].axis("off")

    # Карта residuals в последнем ряду
    ax_map = axes[-cols]
    valid = [p for p in points if p.get("dem_egm_center") is not None]
    fit_pts = {pp["name"]: pp for pp in fit["per_point"]}
    cmap = plt.cm.RdBu_r
    for p in valid:
        fp = fit_pts.get(p["name"])
        if not fp:
            continue
        color = cmap((fp["residual"] + RANSAC_THRESHOLD_M)
                     / (2 * RANSAC_THRESHOLD_M))
        m = "o" if fp["kept"] else "X"
        ax_map.scatter(p["x_m"], p["y_m"], c=[color], s=200, marker=m,
                       edgecolors="black", linewidths=1.5)
        ax_map.annotate(p["name"], (p["x_m"], p["y_m"]), fontsize=7,
                        xytext=(7, 7), textcoords="offset points")
    ax_map.set_title(f"Residuals на карте (✓ kept, ✗ rejected)\n"
                     f"RMS={fit['rms_m']} м", fontsize=10)
    ax_map.set_xlabel("X UTM (м)")
    ax_map.set_ylabel("Y UTM (м)")
    ax_map.set_aspect("equal")
    ax_map.grid(alpha=0.3)
    # Скрыть остальные на последнем ряду
    for j in range(1, cols):
        axes[-cols + j].axis("off")

    fig.tight_layout()
    fig.savefig(OUT_DIAG_PNG, dpi=120, bbox_inches="tight")
    plt.close(fig)
    print(f"  ✓ диагностический PNG: {OUT_DIAG_PNG}")

    # === Сохраняем JSON ===
    serializable_points = []
    for p in points:
        out = {k: v for k, v in p.items() if k not in ("tile", "tile_origin")}
        serializable_points.append(out)
    report = {
        "n_points_input": len(ALL_POINTS),
        "fit": {k: v for k, v in fit.items() if k != "iterations"},
        "fit_iterations": fit["iterations"],
        "loo_validation": loo,
        "diagnostics": serializable_points,
    }
    OUT_REPORT.write_text(json.dumps(report, indent=2, ensure_ascii=False,
                                     default=lambda o: float(o)
                                     if hasattr(o, "item") else str(o)))
    print(f"  ✓ JSON: {OUT_REPORT}")

    # === ИТОГ ===
    print(f"\n{'═' * 72}")
    print(f"  ИТОГ")
    print(f"{'═' * 72}")
    print(f"  По 15 точкам: {fit['n_kept']} прошли, {fit['n_total']-fit['n_kept']} выбросов")
    print(f"  Подгонка:    offset = {fit['a_offset_at_x0y0']:+.3f} м")
    print(f"               tilt = {fit['b_per_km']:+.3f} м/км по X, "
          f"{fit['c_per_km']:+.3f} м/км по Y")
    print(f"  RMS:         {fit['rms_m']} м")
    if not loo.get("skipped"):
        print(f"  LOO RMS:     {loo['rms_m']} м (ожидаемая ошибка в "
              f"типичной точке)")
    print(f"  → DEM в БС-77: {DEM_BS77_OUT}")


if __name__ == "__main__":
    main()

  Калибровка по 15 паспортным точкам (4 старых + 11 новых)
DEM: spb_dem/dem_egm2008.tif, 52224×33895

Читаю ATL08...
  43,334 точек

────────────────────────────────────────────────────────────────────────
  ДИАГНОСТИКА ПО КАЖДОЙ ТОЧКЕ
────────────────────────────────────────────────────────────────────────
  name       h_bs77   DEM_egm    std  ∆max  ATL_dz expected_offset
  Gorny       +2.85    +3.678   7.95  30.9   -1.25          -0.832 ⚠
  KZS_01      +4.35    +2.856   2.06  15.9   +0.62          +1.491 ⚠
  KZS_02      +3.37    +1.659   1.62  17.7   +0.61          +1.708 ⚠
  KZS_03      +3.77    +0.182   3.37  20.2   +0.25          +3.588 ⚠
  P01         +3.00    -0.664   2.80  17.7   +0.24          +3.664 ⚠
  P02         +4.91    +2.319   2.04  13.9   +0.29          +2.591 ⚠
  P03         +4.87    +1.170   2.02  13.7   +0.29          +3.700 ⚠
  P04         +3.00    -0.844   3.06  18.9   +0.24          +3.844 ⚠
  P05         +2.65    +1.414   2.50  24.1   +0.61          +1.236 ⚠
  P

In [10]:
#!/usr/bin/env python3
"""
[1/3] Генерация OSM-маски зданий для всего AOI в виде растра, выровненного
с ArcticDEM эллипсоидальным DEM.

Что делает:
  - Скачивает все OSM building-полигоны в bbox СПб через osmnx
  - Дополнительно: в исторической застройке (Адмиралтейский, Васильевский,
    Петроградка, Центральный) с плотной планировкой добавляет растровую
    диалатацию, чтобы захватить тонкие фрагменты между зданиями
  - Растеризует в маску 1=здание, 0=земля, выравнивает с DEM_ell
  - Сохраняет как COG для эффективного чтения по тайлам в шаге 2

Выход: spb_dem/building_mask.tif (uint8, 0/1)

Запуск:
    python 1_osm_buildings.py
"""

from __future__ import annotations

import sys
from pathlib import Path

import numpy as np
import geopandas as gpd
import rasterio
from rasterio.features import rasterize
from rasterio.windows import Window
from shapely.geometry import box, MultiPolygon, Polygon
from shapely.ops import unary_union


WORK_DIR = Path("./spb_dem")
DEM_ELL = WORK_DIR / "dem_ellipsoidal.tif"
OUT_MASK = WORK_DIR / "building_mask.tif"

# AOI как в основном пайплайне
AOI_BBOX_WGS84 = (29.5, 59.7, 30.7, 60.2)

# Историческая плотная застройка СПб (lon_min, lat_min, lon_max, lat_max).
# В этих зонах применяем дополнительную дилатацию маски на 1 пиксель,
# чтобы захватить тонкие участки между зданиями, которые SMRF потом
# не сможет правильно отфильтровать.
HISTORIC_AREAS = [
    ("Адмиралтейский_центр", (30.30, 59.92, 30.36, 59.95)),
    ("Васильевский",         (30.24, 59.93, 30.30, 59.96)),
    ("Петроградка",          (30.27, 59.95, 30.34, 59.98)),
    ("Центральный",          (30.32, 59.92, 30.40, 59.95)),
]

# Дилатация в исторической зоне (пикселей)
HISTORIC_DILATE_PX = 2

# Растеризация: chunk size в пикселях DEM
CHUNK_SIZE = 4096


# =============================================================================

def fetch_buildings() -> gpd.GeoDataFrame:
    """Скачать все OSM-здания в AOI через osmnx."""
    import osmnx as ox

    ox.settings.timeout = 300
    ox.settings.requests_timeout = 300

    print(f"OSM building запрос для bbox={AOI_BBOX_WGS84}")
    print("  (может занять 5-15 минут в плотных районах)")

    g = ox.features_from_bbox(bbox=AOI_BBOX_WGS84, tags={"building": True})
    g = g[g.geometry.type.isin(["Polygon", "MultiPolygon"])]
    g = g[["geometry"]].to_crs(4326)

    print(f"  получено {len(g):,} полигонов")
    if not len(g):
        sys.exit("Нет полигонов — проверьте сеть и bbox")

    # Сразу удаляем пустые геометрии и невалидные
    g = g[g.geometry.is_valid]
    g = g[~g.geometry.is_empty]
    print(f"  после очистки невалидных: {len(g):,}")

    return g


def rasterize_in_chunks(buildings: gpd.GeoDataFrame, dem_path: Path,
                         out_path: Path) -> None:
    """
    Растеризуем здания в COG с тем же transform/crs что и DEM, чанками
    чтобы не загружать растр в RAM целиком.
    """
    with rasterio.open(dem_path) as src:
        profile = src.profile.copy()
        profile.update(
            dtype="uint8", count=1, compress="DEFLATE", predictor=2,
            tiled=True, blockxsize=512, blockysize=512,
            BIGTIFF="IF_SAFER", nodata=255,
        )
        # Перепроецируем здания в CRS растра один раз
        bld_in_crs = buildings.to_crs(src.crs)
        # Для ускорения rtree-индекс
        bld_sindex = bld_in_crs.sindex

        # Историческая зона маска (в CRS растра)
        historic_geoms = []
        from shapely.geometry import box as sbox
        for _, (lo1, la1, lo2, la2) in HISTORIC_AREAS:
            historic_geoms.append(sbox(lo1, la1, lo2, la2))
        historic_union_wgs = unary_union(historic_geoms) if historic_geoms else None
        if historic_union_wgs is not None:
            historic_gdf = gpd.GeoDataFrame(
                geometry=[historic_union_wgs], crs=4326).to_crs(src.crs)
            historic_geom_crs = historic_gdf.geometry.iloc[0]
        else:
            historic_geom_crs = None

        with rasterio.open(out_path, "w", **profile) as dst:
            n_chunks_x = (src.width + CHUNK_SIZE - 1) // CHUNK_SIZE
            n_chunks_y = (src.height + CHUNK_SIZE - 1) // CHUNK_SIZE
            n_total = n_chunks_x * n_chunks_y
            n_done = 0

            for cy in range(n_chunks_y):
                for cx in range(n_chunks_x):
                    n_done += 1
                    r0, c0 = cy * CHUNK_SIZE, cx * CHUNK_SIZE
                    h = min(CHUNK_SIZE, src.height - r0)
                    w = min(CHUNK_SIZE, src.width - c0)
                    win = Window(c0, r0, w, h)

                    # Bbox чанка в CRS растра
                    win_xmin = src.transform.c + c0 * src.transform.a
                    win_ymax = src.transform.f + r0 * src.transform.e
                    win_xmax = win_xmin + w * src.transform.a
                    win_ymin = win_ymax + h * src.transform.e
                    win_box = box(min(win_xmin, win_xmax),
                                  min(win_ymin, win_ymax),
                                  max(win_xmin, win_xmax),
                                  max(win_ymin, win_ymax))

                    # Выбираем здания, пересекающие чанк
                    candidates_idx = list(bld_sindex.intersection(win_box.bounds))
                    if not candidates_idx:
                        # Пустой чанк — пишем нули
                        chunk_mask = np.zeros((h, w), dtype=np.uint8)
                        dst.write(chunk_mask, 1, window=win)
                        if n_done % max(1, n_total // 20) == 0 or n_done == n_total:
                            print(f"  чанк {n_done}/{n_total} ({100*n_done/n_total:.0f}%)")
                        continue

                    candidates = bld_in_crs.iloc[candidates_idx]
                    # Реальное пересечение
                    candidates = candidates[candidates.geometry.intersects(win_box)]
                    if candidates.empty:
                        chunk_mask = np.zeros((h, w), dtype=np.uint8)
                        dst.write(chunk_mask, 1, window=win)
                        continue

                    # Transform для этого чанка
                    chunk_transform = rasterio.windows.transform(win, src.transform)

                    # Растеризуем
                    chunk_mask = rasterize(
                        [(geom, 1) for geom in candidates.geometry],
                        out_shape=(h, w),
                        transform=chunk_transform,
                        fill=0,
                        dtype=np.uint8,
                        all_touched=True,  # ловим краевые пиксели
                    )

                    # Дилатация в исторической зоне
                    if historic_geom_crs is not None and chunk_mask.any():
                        if win_box.intersects(historic_geom_crs):
                            from scipy.ndimage import binary_dilation
                            in_historic = rasterize(
                                [(historic_geom_crs.intersection(win_box), 1)],
                                out_shape=(h, w),
                                transform=chunk_transform,
                                fill=0, dtype=np.uint8,
                            )
                            dilated = binary_dilation(
                                chunk_mask, iterations=HISTORIC_DILATE_PX
                            ).astype(np.uint8)
                            # В исторической зоне берём дилатированную, иначе — оригинал
                            chunk_mask = np.where(in_historic == 1,
                                                  dilated, chunk_mask)

                    dst.write(chunk_mask, 1, window=win)

                    if n_done % max(1, n_total // 20) == 0 or n_done == n_total:
                        n_buildings = (chunk_mask > 0).sum()
                        print(f"  чанк {n_done}/{n_total} "
                              f"({100*n_done/n_total:.0f}%), "
                              f"{n_buildings:,} px зданий в чанке")


def main() -> None:
    if not DEM_ELL.exists():
        sys.exit(f"Нет {DEM_ELL} — сначала запустите spb_dem_build_v2.py")

    print("=" * 70)
    print("  [1/3] Генерация OSM building маски")
    print("=" * 70)

    if OUT_MASK.exists():
        print(f"⚠ {OUT_MASK} уже существует. Удалите для пересоздания.")
        return

    print("\nШаг A: запрос OSM-зданий")
    buildings = fetch_buildings()

    # Сохраняем GeoPackage сырых полигонов на случай ручного контроля
    raw_gpkg = WORK_DIR / "osm_buildings_raw.gpkg"
    if not raw_gpkg.exists():
        try:
            buildings.to_file(raw_gpkg, driver="GPKG", layer="buildings")
            print(f"  ✓ полигоны сохранены: {raw_gpkg}")
        except Exception as e:
            print(f"  ! не удалось сохранить полигоны: {e}")

    print("\nШаг B: растеризация в COG (тайл 2 м)")
    rasterize_in_chunks(buildings, DEM_ELL, OUT_MASK)

    # Финальная статистика
    with rasterio.open(OUT_MASK) as src:
        # Подсчёт по тайлам, не загружая всё
        n_buildings_px = 0
        n_total_px = 0
        for _, win in src.block_windows(1):
            arr = src.read(1, window=win)
            n_buildings_px += int((arr == 1).sum())
            n_total_px += int(arr.size)
        frac = n_buildings_px / n_total_px

    print(f"\n✓ Готово: {OUT_MASK}")
    print(f"  всего пикселей: {n_total_px:,}")
    print(f"  пикселей зданий: {n_buildings_px:,} ({100*frac:.1f}%)")


if __name__ == "__main__":
    main()

  [1/3] Генерация OSM building маски

Шаг A: запрос OSM-зданий
OSM building запрос для bbox=(29.5, 59.7, 30.7, 60.2)
  (может занять 5-15 минут в плотных районах)
  получено 299,819 полигонов
  после очистки невалидных: 299,819
  ✓ полигоны сохранены: spb_dem/osm_buildings_raw.gpkg

Шаг B: растеризация в COG (тайл 2 м)
  чанк 5/117 (4%), 319,248 px зданий в чанке
  чанк 10/117 (9%), 782 px зданий в чанке
  чанк 15/117 (13%), 213 px зданий в чанке
  чанк 20/117 (17%), 964,967 px зданий в чанке
  чанк 25/117 (21%)
  чанк 30/117 (26%), 320,341 px зданий в чанке
  чанк 35/117 (30%), 395,391 px зданий в чанке
  чанк 40/117 (34%)
  чанк 45/117 (38%)
  чанк 50/117 (43%)
  чанк 55/117 (47%), 38,398 px зданий в чанке
  чанк 60/117 (51%), 3,276,209 px зданий в чанке
  чанк 65/117 (56%)
  чанк 70/117 (60%), 498,083 px зданий в чанке
  чанк 75/117 (64%), 239,952 px зданий в чанке
  чанк 80/117 (68%), 33,681 px зданий в чанке
  чанк 85/117 (73%), 329,959 px зданий в чанке
  чанк 90/117 (77%)
  чанк

In [11]:
#!/usr/bin/env python3
"""
[2/3] SMRF bare-earth фильтрация ArcticDEM с тайловой обработкой
и weighted blend stitching.

Параметры по Liu et al. NHESS 2023 (Helsinki, прямой аналог СПб):
    slope = 0.07 (max terrain slope)
    window_max = 30 m
    threshold = 0.5 m
    scaling = 1.25
    cell = 2 m

Архитектура:
  - Тайлы 2 km × 2 km (1000×1000 px), overlap 100 m с каждой стороны
  - Каждый тайл фильтруется независимо
  - Stitching через distance-to-edge weighting в overlap-зоне
  - OSM-маска зданий применяется ДО SMRF (заполняется через ndimage.distance_transform)
  - Финальный DTM_ell в полном разрешении 2 м

Чтобы умещаться в RAM:
  - Один тайл: 1100×1100 × 4 байта = 5 МБ для DEM + 1 МБ для маски + ~30 МБ
    для промежуточных операций SMRF = ~40 МБ peak.
  - Финальный растр пишем по тайлам в COG, не держим в памяти.

Запуск:
    python 2_smrf_filter.py
"""

from __future__ import annotations

import gc
import sys
from pathlib import Path
from typing import Iterable

import numpy as np
import rasterio
from rasterio.windows import Window
from scipy.ndimage import (
    binary_dilation, binary_erosion, generate_binary_structure,
    grey_erosion, grey_dilation,
)


WORK_DIR = Path("./spb_dem")
DEM_ELL = WORK_DIR / "dem_ellipsoidal.tif"
BUILDING_MASK = WORK_DIR / "building_mask.tif"
DTM_ELL = WORK_DIR / "dtm_ellipsoidal.tif"     # выход

# --- Параметры SMRF (Liu 2023) ----------------------------------------------
SMRF_SLOPE = 0.07          # max ground slope (радиансы tan)
SMRF_WIN_MAX_M = 30.0      # максимальное окно фильтра, м
SMRF_THRESHOLD_M = 0.5     # порог для классификации
SMRF_SCALING = 1.25        # коэффициент роста окна
SMRF_CELL_SIZE = 2.0       # размер пикселя растра, м

# --- Тайлинг ----------------------------------------------------------------
TILE_INNER_M = 2000.0      # размер inner-зоны тайла, м (= 1000 px)
TILE_OVERLAP_M = 100.0     # overlap, м (= 50 px)


# =============================================================================
#                            ЯДРО SMRF (Pingel 2013)
# =============================================================================

def smrf_classify(dem: np.ndarray, cell_size: float, slope: float,
                  win_max_m: float, threshold: float,
                  scaling: float = 1.25,
                  building_mask: np.ndarray | None = None,
                  ) -> tuple[np.ndarray, np.ndarray]:
    """
    Simple Morphological Filter (Pingel et al. 2013).
    Возвращает (object_mask, dtm) где
       object_mask[i,j] == True: пиксель — объект (здание/дерево)
       dtm[i,j]: высота земли (NaN где не удалось восстановить)

    Если передана building_mask — здания сразу помечены как objects
    и не используются в SMRF-итерациях.
    """
    h = dem.astype(np.float32, copy=True)
    nodata_mask = ~np.isfinite(h)

    if building_mask is not None:
        h_for_smrf = h.copy()
        # Здания заменяем на временный максимум окружения, чтобы они не
        # тянули SMRF вниз. После итераций восстановим как objects.
        h_for_smrf[building_mask] = np.nan
        # Заполнение через большое morphological dilation сверху
        # (быстро и устойчиво)
        from scipy.ndimage import generic_filter
        # Простая стратегия: NaN заменяем медианой 21×21 окрестности
        valid = np.isfinite(h_for_smrf)
        if valid.sum() > 0:
            # Iterative fill: каждый шаг расширяет valid через max
            filled = h_for_smrf.copy()
            for _ in range(5):
                holes = ~np.isfinite(filled)
                if not holes.any():
                    break
                dilated = grey_dilation(
                    np.where(holes, -np.inf, filled), size=5)
                filled = np.where(holes, dilated, filled)
                # Очистим -inf, если остались
                filled[~np.isfinite(filled)] = np.nanmedian(filled[np.isfinite(filled)])
            h = filled
        else:
            pass  # нет валидных — просто оставим nodata
    
    # SMRF iterations
    h_filtered = h.copy()
    h_filtered[nodata_mask] = np.nanmin(h[~nodata_mask]) if (~nodata_mask).any() else 0
    
    object_mask = nodata_mask.copy()
    
    win_max_px = int(win_max_m / cell_size)
    
    # Прогрессивный morphological opening (erode then dilate) с растущим окном
    win = 1
    iteration = 0
    while win <= win_max_px:
        size = 2 * win + 1
        # Opening = erosion затем dilation
        eroded = grey_erosion(h_filtered, size=size, mode='reflect')
        opened = grey_dilation(eroded, size=size, mode='reflect')
        
        # Slope threshold для этой итерации
        # T_w = slope * win_size_m + threshold
        win_size_m = win * cell_size
        t_w = slope * win_size_m + threshold
        
        # Помечаем как объект пиксели, где h - opened > t_w
        diff = h_filtered - opened
        new_objects = diff > t_w
        object_mask |= new_objects
        
        # Заменяем найденные объекты их opened-версиями для следующей итерации
        h_filtered = np.where(new_objects, opened, h_filtered)
        
        # Растим окно
        win = max(win + 1, int(win * scaling))
        iteration += 1
        
        # Safety break
        if iteration > 30:
            break
    
    # Если building_mask был, добавим здания в objects
    if building_mask is not None:
        object_mask |= building_mask
    
    # DTM: только не-объекты
    dtm = np.where(object_mask | nodata_mask, np.nan, h_filtered)
    return object_mask, dtm


def inpaint_dtm(dtm: np.ndarray, max_distance_m: float = 50.0,
                cell_size: float = 2.0) -> np.ndarray:
    """
    Восстанавливаем земные значения в области objects через intepolation
    от ближайших валидных. Используем Scipy distance_transform_edt
    с return_indices.
    """
    from scipy.ndimage import distance_transform_edt
    
    valid_mask = np.isfinite(dtm)
    if not valid_mask.any():
        return dtm
    if valid_mask.all():
        return dtm
    
    # distance_transform_edt с indices: для каждого invalid пикселя возвращает
    # индексы ближайшего valid
    dist, (idx_r, idx_c) = distance_transform_edt(
        ~valid_mask, return_indices=True, sampling=cell_size,
    )
    
    # Заполняем
    out = dtm.copy()
    fill_mask = ~valid_mask
    out[fill_mask] = dtm[idx_r[fill_mask], idx_c[fill_mask]]
    
    # Где расстояние слишком большое (например, центр большого здания), 
    # все равно заполнили, но можем флагнуть. Для фрод-карты это OK.
    if max_distance_m > 0:
        far = dist > max_distance_m
        if far.any():
            print(f"      ⚠ {far.sum():,} пикселей с inpaint dist > "
                  f"{max_distance_m} м — возможна неточность")
    
    return out


# =============================================================================
#                             ТАЙЛОВЫЙ ПРОЦЕССОР
# =============================================================================

def make_blend_weights(h: int, w: int, overlap_h: int, overlap_w: int) -> np.ndarray:
    """
    Веса для weighted blend в overlap-зоне.
    Center пиксели = 1.0, по краям overlap-зоны линейно падают до 0.
    """
    weight_y = np.ones(h, dtype=np.float32)
    weight_x = np.ones(w, dtype=np.float32)
    
    if overlap_h > 0:
        ramp_in = np.linspace(0, 1, overlap_h, endpoint=False, dtype=np.float32)
        ramp_out = ramp_in[::-1]
        weight_y[:overlap_h] = ramp_in
        weight_y[-overlap_h:] = ramp_out
    
    if overlap_w > 0:
        ramp_in = np.linspace(0, 1, overlap_w, endpoint=False, dtype=np.float32)
        ramp_out = ramp_in[::-1]
        weight_x[:overlap_w] = ramp_in
        weight_x[-overlap_w:] = ramp_out
    
    return weight_y[:, None] * weight_x[None, :]


def process_one_tile(
    src_dem: rasterio.DatasetReader,
    src_mask: rasterio.DatasetReader,
    inner_window: Window,
    overlap_px: int,
    dem_nodata: float,
) -> tuple[np.ndarray, Window]:
    """
    Прочитать тайл с overlap, фильтровать SMRF + inpaint, вернуть
    DTM и расширенный window (с overlap).
    """
    expanded = Window(
        col_off=max(0, inner_window.col_off - overlap_px),
        row_off=max(0, inner_window.row_off - overlap_px),
        width=min(src_dem.width - max(0, inner_window.col_off - overlap_px),
                   inner_window.width + 2 * overlap_px),
        height=min(src_dem.height - max(0, inner_window.row_off - overlap_px),
                    inner_window.height + 2 * overlap_px),
    )
    
    # Читаем DEM
    dem = src_dem.read(1, window=expanded).astype(np.float32)
    if dem_nodata is not None:
        dem[dem == dem_nodata] = np.nan
    
    # Читаем маску зданий (если выровнены)
    mask = None
    if src_mask is not None:
        mask_arr = src_mask.read(1, window=expanded)
        mask = (mask_arr == 1)
    
    # SMRF
    object_mask, dtm = smrf_classify(
        dem, cell_size=SMRF_CELL_SIZE,
        slope=SMRF_SLOPE,
        win_max_m=SMRF_WIN_MAX_M,
        threshold=SMRF_THRESHOLD_M,
        scaling=SMRF_SCALING,
        building_mask=mask,
    )
    
    # Inpaint
    dtm_filled = inpaint_dtm(dtm, max_distance_m=50.0, cell_size=SMRF_CELL_SIZE)
    
    return dtm_filled, expanded


# =============================================================================
#                                   MAIN
# =============================================================================

def main() -> None:
    if not DEM_ELL.exists():
        sys.exit(f"Нет {DEM_ELL}")
    if not BUILDING_MASK.exists():
        sys.exit(f"Нет {BUILDING_MASK} — сначала запустите 1_osm_buildings.py")
    
    print("=" * 70)
    print("  [2/3] SMRF bare-earth filter (tile-based)")
    print("=" * 70)
    print(f"  Параметры: slope={SMRF_SLOPE}, window_max={SMRF_WIN_MAX_M}м, "
          f"thr={SMRF_THRESHOLD_M}м (Liu 2023 Helsinki)")
    print(f"  Тайл inner: {TILE_INNER_M:.0f}м, overlap: {TILE_OVERLAP_M:.0f}м")

    if DTM_ELL.exists():
        print(f"⚠ {DTM_ELL} уже существует. Удалите для пересоздания.")
        return

    src_dem = rasterio.open(DEM_ELL)
    src_mask = rasterio.open(BUILDING_MASK)
    
    cell_size = abs(src_dem.transform.a)
    inner_px = int(round(TILE_INNER_M / cell_size))
    overlap_px = int(round(TILE_OVERLAP_M / cell_size))
    
    print(f"  Размер растра: {src_dem.width} × {src_dem.height} px "
          f"({src_dem.width * cell_size / 1000:.1f} × "
          f"{src_dem.height * cell_size / 1000:.1f} км)")
    print(f"  Тайл inner: {inner_px} px, overlap: {overlap_px} px")
    
    n_tiles_x = (src_dem.width + inner_px - 1) // inner_px
    n_tiles_y = (src_dem.height + inner_px - 1) // inner_px
    n_total = n_tiles_x * n_tiles_y
    print(f"  Всего тайлов: {n_tiles_x} × {n_tiles_y} = {n_total}")
    
    # Профиль выходного DTM
    profile = src_dem.profile.copy()
    profile.update(
        dtype="float32", compress="DEFLATE", predictor=3,
        tiled=True, blockxsize=512, blockysize=512,
        BIGTIFF="IF_SAFER", nodata=-9999.0,
    )
    
    # Промежуточные накопители для blend stitching: weighted_sum и weight_sum
    # Их нельзя держать в RAM целиком — пишем в два промежуточных растра
    accum_path = WORK_DIR / "_smrf_accum_sum.tif"
    weight_path = WORK_DIR / "_smrf_accum_weight.tif"
    
    accum_profile = profile.copy()
    weight_profile = profile.copy()
    
    # Инициализируем нулями
    print(f"\n  Инициализация промежуточных растров для blend...")
    with rasterio.open(accum_path, "w", **accum_profile) as dst_a:
        for _, win in dst_a.block_windows(1):
            dst_a.write(np.zeros((win.height, win.width), dtype=np.float32),
                        1, window=win)
    with rasterio.open(weight_path, "w", **weight_profile) as dst_w:
        for _, win in dst_w.block_windows(1):
            dst_w.write(np.zeros((win.height, win.width), dtype=np.float32),
                        1, window=win)
    
    dem_nodata = src_dem.nodata
    
    # --- Главный цикл по тайлам ---------------------------------------------
    print(f"\n  Обрабатываем тайлы...")
    n_done = 0
    accum = rasterio.open(accum_path, "r+")
    weights = rasterio.open(weight_path, "r+")
    
    try:
        for ty in range(n_tiles_y):
            for tx in range(n_tiles_x):
                n_done += 1
                
                inner_c0 = tx * inner_px
                inner_r0 = ty * inner_px
                inner_w = min(inner_px, src_dem.width - inner_c0)
                inner_h = min(inner_px, src_dem.height - inner_r0)
                inner_win = Window(inner_c0, inner_r0, inner_w, inner_h)
                
                try:
                    dtm_tile, expanded_win = process_one_tile(
                        src_dem, src_mask, inner_win, overlap_px, dem_nodata)
                except Exception as e:
                    print(f"    ! тайл {n_done}/{n_total} (tx={tx},ty={ty}): {e}")
                    continue
                
                # Веса blend-зоны
                # Для тайлов на краю растра overlap может быть меньше с одной стороны
                actual_overlap_top = inner_r0 - expanded_win.row_off
                actual_overlap_left = inner_c0 - expanded_win.col_off
                actual_overlap_bottom = (expanded_win.row_off + expanded_win.height
                                          - (inner_r0 + inner_h))
                actual_overlap_right = (expanded_win.col_off + expanded_win.width
                                         - (inner_c0 + inner_w))
                
                # Создаём пирамидальный вес: симметричный ramp на overlap-зонах
                weight = np.ones((expanded_win.height, expanded_win.width),
                                  dtype=np.float32)
                if actual_overlap_top > 0:
                    ramp = np.linspace(0, 1, actual_overlap_top,
                                        endpoint=False, dtype=np.float32)
                    weight[:actual_overlap_top, :] *= ramp[:, None]
                if actual_overlap_bottom > 0:
                    ramp = np.linspace(1, 0, actual_overlap_bottom,
                                        endpoint=False, dtype=np.float32)
                    weight[-actual_overlap_bottom:, :] *= ramp[:, None]
                if actual_overlap_left > 0:
                    ramp = np.linspace(0, 1, actual_overlap_left,
                                        endpoint=False, dtype=np.float32)
                    weight[:, :actual_overlap_left] *= ramp[None, :]
                if actual_overlap_right > 0:
                    ramp = np.linspace(1, 0, actual_overlap_right,
                                        endpoint=False, dtype=np.float32)
                    weight[:, -actual_overlap_right:] *= ramp[None, :]
                
                # Где DTM NaN, вес = 0
                dtm_valid = np.isfinite(dtm_tile)
                weight = np.where(dtm_valid, weight, 0.0)
                dtm_for_accum = np.where(dtm_valid, dtm_tile, 0.0).astype(np.float32)
                
                # Читаем существующие значения и добавляем
                cur_a = accum.read(1, window=expanded_win)
                cur_w = weights.read(1, window=expanded_win)
                
                accum.write(cur_a + dtm_for_accum * weight, 1, window=expanded_win)
                weights.write(cur_w + weight, 1, window=expanded_win)
                
                if n_done % max(1, n_total // 30) == 0 or n_done == n_total:
                    print(f"    тайл {n_done}/{n_total} ({100*n_done/n_total:.0f}%)")
                
                # Cleanup
                del dtm_tile, weight, dtm_for_accum, cur_a, cur_w
                gc.collect()
    finally:
        accum.close()
        weights.close()
    
    # --- Финальное деление: DTM = accum / weight, чанками ------------------
    print(f"\n  Финал: DTM = sum / weight, тайлово")
    with rasterio.open(accum_path) as a, \
         rasterio.open(weight_path) as w, \
         rasterio.open(DTM_ELL, "w", **profile) as dst:
        for _, win in dst.block_windows(1):
            sum_arr = a.read(1, window=win)
            w_arr = w.read(1, window=win)
            with np.errstate(invalid='ignore', divide='ignore'):
                dtm = np.where(w_arr > 0, sum_arr / w_arr, profile['nodata'])
            dst.write(dtm.astype(np.float32), 1, window=win)
    
    # Удаляем промежуточные
    print(f"\n  Удаление промежуточных файлов...")
    accum_path.unlink()
    weight_path.unlink()
    
    src_dem.close()
    src_mask.close()
    
    print(f"\n✓ Готово: {DTM_ELL}")


if __name__ == "__main__":
    main()

  [2/3] SMRF bare-earth filter (tile-based)
  Параметры: slope=0.07, window_max=30.0м, thr=0.5м (Liu 2023 Helsinki)
  Тайл inner: 2000м, overlap: 100м
  Размер растра: 52224 × 33895 px (104.4 × 67.8 км)
  Тайл inner: 1000 px, overlap: 50 px
  Всего тайлов: 53 × 34 = 1802

  Инициализация промежуточных растров для blend...

  Обрабатываем тайлы...
      ⚠ 1,209,432 пикселей с inpaint dist > 50.0 м — возможна неточность
      ⚠ 1,079,843 пикселей с inpaint dist > 50.0 м — возможна неточность
      ⚠ 836,327 пикселей с inpaint dist > 50.0 м — возможна неточность
      ⚠ 599,751 пикселей с inpaint dist > 50.0 м — возможна неточность
      ⚠ 361,691 пикселей с inpaint dist > 50.0 м — возможна неточность
      ⚠ 118,778 пикселей с inpaint dist > 50.0 м — возможна неточность
      ⚠ 887,949 пикселей с inpaint dist > 50.0 м — возможна неточность
    тайл 60/1802 (3%)
      ⚠ 1,187,725 пикселей с inpaint dist > 50.0 м — возможна неточность
      ⚠ 996,204 пикселей с inpaint dist > 50.0 м — возм

In [12]:
#!/usr/bin/env python3
"""
[3/3] Финальная калибровка DTM эллипсоид → БС-77.

Принципы:
  1. ICESat-2 переcсэмплируем на DTM (НЕ на DSM!) — теперь dZ = ATL08 - DTM
     должен быть chistеr, потому что обе поверхности — земля.
  2. Только КОНСТАНТНЫЙ сдвиг — никаких плоскостей. Параметры b, c
     нефизичны при квазигеоидном градиенте < 0.05 м/км для AOI 30 км.
  3. Калибровка ТОЛЬКО на «сертифицированно плоских» паспортных точках:
       - std в окне 21×21 < 0.3 м (вокруг точки реальный плоский рельеф)
       - dZ_ICESat_MAD < 0.3 м (DEM локально подтверждается ICESat-2)
     Это автоматически отсеет точки на парапетах, мачтах, насыпях.
  4. LOO-валидация — обязательный гейт. RMS > 0.5 м = тревога.

Запуск:
    python 3_calibrate.py
"""

from __future__ import annotations

import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.windows import Window
import matplotlib.pyplot as plt
from pyproj import Transformer
from scipy.spatial import cKDTree
from scipy import stats


WORK_DIR = Path("./spb_dem")
DTM_ELL = WORK_DIR / "dtm_ellipsoidal.tif"
ATL08_GPKG = WORK_DIR / "atl08_ground.gpkg"
DTM_BS77 = WORK_DIR / "dtm_bs77.tif"

OUT_REPORT = WORK_DIR / "calibration_v3.json"
OUT_PNG = WORK_DIR / "calibration_v3.png"

WORK_EPSG = 32636

# --- Все 15 паспортных точек -----------------------------------------------
ALL_POINTS = [
    ("Gorny",     30.269963, 59.929175, 2.846, "old"),
    ("KZS_01",    29.694830, 59.994314, 4.347, "old"),
    ("KZS_02",    29.696649, 59.987700, 3.367, "old"),
    ("KZS_03",    29.836873, 60.027940, 3.770, "old"),
    ("P01",       29.842406, 60.027048, 3.00,  "new"),
    ("P02",       29.845407, 60.026729, 4.91,  "new"),
    ("P03",       29.845589, 60.026572, 4.87,  "new"),
    ("P04",       29.841349, 60.026912, 3.00,  "new"),
    ("P05",       29.698166, 59.992611, 2.65,  "new"),
    ("P06",       29.835426, 60.025722, 3.25,  "new"),
    ("P07",       29.696296, 59.989239, 3.00,  "new"),
    ("P08",       29.721013, 60.012491, 3.20,  "new"),
    ("P09",       30.108990, 60.019754, 5.27,  "new"),
    ("P10",       30.110281, 60.019732, 4.77,  "new"),
    ("P11",       30.214238, 59.878634, 3.50,  "new"),
]

# --- Критерии «плоской» точки -----------------------------------------------
WIN_PX = 21                          # окно 42×42 м для оценки плоскости
PASS_DEM_STD_MAX_M = 0.3             # std в окне DTM ≤ этого
PASS_ICESAT_MAD_MAX_M = 0.3          # dZ_MAD по 30 ближайших ATL08
PASS_ICESAT_MIN_PTS = 5              # минимум ATL08 точек в радиусе


# =============================================================================
#                ШАГ 1: РЕЗАМПЛ ATL08 НА DTM (не на DSM)
# =============================================================================

def resample_atl08_on_dtm(dtm_path: Path,
                          atl08: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    """
    Прежде ATL08 был сэмплирован на DSM. Теперь нужно пересэмплировать на DTM,
    чтобы dZ_DTM = ATL08 - DTM_ell отражало остаточную ошибку коррегистрации.
    """
    with rasterio.open(dtm_path) as src:
        gdf = atl08.copy()
        if gdf.crs is None or gdf.crs.to_epsg() != src.crs.to_epsg():
            gdf_dem = gdf.to_crs(src.crs)
        else:
            gdf_dem = gdf
        xy = [(p.x, p.y) for p in gdf_dem.geometry]
        vals = np.array([v[0] for v in src.sample(xy)], dtype=float)
        if src.nodata is not None:
            vals[vals == src.nodata] = np.nan
        vals[vals < -1000] = np.nan

    gdf["h_dtm_ell"] = vals
    gdf["dZ_dtm"] = gdf["h_te_ell"] - gdf["h_dtm_ell"]
    gdf = gdf[np.isfinite(gdf["dZ_dtm"]) & (gdf["dZ_dtm"].abs() < 5)].copy()
    return gdf


# =============================================================================
#         ШАГ 2: КОРРЕГИСТРАЦИЯ DTM ПО ICESat-2 (только сдвиг)
# =============================================================================

def coregister_constant_shift(atl08_with_dtm: gpd.GeoDataFrame) -> dict:
    """
    Только медиана dZ как сдвиг. Без плоскости.
    """
    z = atl08_with_dtm["dZ_dtm"].to_numpy()
    median_dz = float(np.median(z))
    mad_dz = float(stats.median_abs_deviation(z, scale="normal"))
    
    # 95% CI медианы через bootstrap
    rng = np.random.default_rng(42)
    n_boot = 500
    medians = []
    for _ in range(n_boot):
        sample = rng.choice(z, size=len(z), replace=True)
        medians.append(float(np.median(sample)))
    medians = np.array(medians)
    ci_lo = float(np.quantile(medians, 0.025))
    ci_hi = float(np.quantile(medians, 0.975))
    
    print(f"  ATL08 ground vs DTM_ell:")
    print(f"    n = {len(z):,} точек")
    print(f"    median dZ = {median_dz:+.3f} м")
    print(f"    MAD       = {mad_dz:.3f} м")
    print(f"    95% CI median = [{ci_lo:+.3f}, {ci_hi:+.3f}] м")
    
    return {
        "n_points": int(len(z)),
        "median_dZ_m": round(median_dz, 4),
        "mad_dZ_m": round(mad_dz, 4),
        "ci_95_lo_m": round(ci_lo, 4),
        "ci_95_hi_m": round(ci_hi, 4),
    }


def apply_constant_shift(dem_in: Path, dem_out: Path, shift: float) -> None:
    """Тайлово прибавить shift."""
    with rasterio.open(dem_in) as src:
        profile = src.profile.copy()
        profile.update(dtype="float32", compress="DEFLATE", predictor=3,
                       BIGTIFF="IF_SAFER")
        nodata = src.nodata
        with rasterio.open(dem_out, "w", **profile) as dst:
            for _, window in src.block_windows(1):
                data = src.read(1, window=window).astype(np.float32)
                out = data + np.float32(shift)
                if nodata is not None:
                    out[data == nodata] = nodata
                dst.write(out, 1, window=window)


# =============================================================================
#              ШАГ 3: ОЦЕНКА «ПЛОСКОСТНОСТИ» ПАСПОРТНЫХ ТОЧЕК
# =============================================================================

def assess_passport_points(dtm_path: Path, atl08: gpd.GeoDataFrame,
                            ) -> list[dict]:
    """
    Для каждой паспортной точки:
      - std в окне 21×21 DTM
      - ICESat-2 dZ статистика в радиусе 500 м
      - вердикт «плоская» (можно использовать для калибровки) или нет
    """
    src = rasterio.open(dtm_path)
    tr = Transformer.from_crs(4326, src.crs, always_xy=True)
    
    atl_xy = atl08[["x_m", "y_m"]].values
    atl_dz_dtm = atl08["dZ_dtm"].values  # после resampling на DTM
    atl_tree = cKDTree(atl_xy)
    
    results = []
    print(f"\n  {'name':<8} {'h_BS77':>8} {'DTM_ell':>9} {'std_DTM':>8} "
          f"{'ATL_n':>5} {'ATL_MAD':>7} {'verdict':>10}")
    
    for name, lon, lat, h_bs77, role in ALL_POINTS:
        x, y = tr.transform(lon, lat)
        
        # DTM окно
        col = (x - src.transform.c) / src.transform.a
        row = (y - src.transform.f) / src.transform.e
        c0 = int(round(col)) - WIN_PX // 2
        r0 = int(round(row)) - WIN_PX // 2
        win = Window(c0, r0, WIN_PX, WIN_PX)
        tile = src.read(1, window=win, boundless=True,
                        fill_value=np.nan).astype(float)
        if src.nodata is not None:
            tile[tile == src.nodata] = np.nan
        valid = tile[np.isfinite(tile)]
        if not valid.size:
            results.append({"name": name, "valid": False, "reason": "NoData"})
            print(f"  {name:<8}  ! NoData")
            continue
        
        center = float(tile[WIN_PX // 2, WIN_PX // 2])
        std = float(np.nanstd(valid))
        med = float(np.nanmedian(valid))
        
        # ATL08 в радиусе 500 м
        idx = atl_tree.query_ball_point([x, y], r=500)
        if len(idx) >= PASS_ICESAT_MIN_PTS:
            atl_local = atl_dz_dtm[idx]
            atl_local = atl_local[np.isfinite(atl_local)]
            if len(atl_local) >= PASS_ICESAT_MIN_PTS:
                atl_mad = float(stats.median_abs_deviation(
                    atl_local, scale="normal"))
                atl_n = len(atl_local)
            else:
                atl_mad = np.nan
                atl_n = len(atl_local)
        else:
            atl_mad = np.nan
            atl_n = len(idx)
        
        # Вердикт
        is_flat_dem = std <= PASS_DEM_STD_MAX_M
        is_flat_atl = (np.isfinite(atl_mad) and atl_mad <= PASS_ICESAT_MAD_MAX_M)
        passes = bool(is_flat_dem and is_flat_atl)
        
        verdict = "✓ flat" if passes else "✗ rough"
        
        # offset_observed = h_BS77 - DTM_ell (если плоская)
        observed_offset = h_bs77 - center if passes else None
        
        rec = {
            "name": name, "role": role, "lon": lon, "lat": lat, "x_m": x, "y_m": y,
            "h_bs77": h_bs77,
            "dtm_ell_center": round(center, 3),
            "dtm_ell_median_window": round(med, 3),
            "dtm_std_window": round(std, 3),
            "atl_dz_n": int(atl_n) if isinstance(atl_n, int) else int(atl_n),
            "atl_dz_mad": round(atl_mad, 3) if np.isfinite(atl_mad) else None,
            "is_flat_dem": is_flat_dem,
            "is_flat_atl": is_flat_atl,
            "passes_filter": passes,
            "observed_offset_bs77_minus_dtm_ell":
                round(observed_offset, 3) if observed_offset is not None else None,
        }
        results.append(rec)
        
        atl_str = f"{atl_mad:.2f}" if np.isfinite(atl_mad) else "n/a"
        print(f"  {name:<8} {h_bs77:>+8.2f} {center:>+9.3f} {std:>8.3f} "
              f"{atl_n:>5d} {atl_str:>7} {verdict:>10}")
    
    src.close()
    return results


# =============================================================================
#         ШАГ 4: ОДИН СДВИГ ЭЛЛИПСОИД → БС-77 ПО ПЛОСКИМ ТОЧКАМ
# =============================================================================

def calibrate_constant_offset(passport_results: list[dict]) -> dict:
    """
    Берём только passes_filter=True точки. Считаем медиану observed_offset.
    LOO-валидация с тем же набором.
    """
    flat = [p for p in passport_results if p.get("passes_filter")]
    if len(flat) < 2:
        sys.exit(f"⚠ Только {len(flat)} плоских точек прошли фильтр. "
                 f"Нужно ≥2.")
    
    offsets = np.array([p["observed_offset_bs77_minus_dtm_ell"] for p in flat])
    names = [p["name"] for p in flat]
    
    print(f"\n  Плоские точки ({len(flat)}/{len(passport_results)}):")
    for p, off in zip(flat, offsets):
        print(f"    {p['name']:<8}: offset = {off:+.3f} м")
    
    median_offset = float(np.median(offsets))
    mad_offset = float(stats.median_abs_deviation(offsets, scale="normal"))
    
    print(f"\n  Медианный сдвиг эллипсоид → БС-77: {median_offset:+.3f} м")
    print(f"  MAD по точкам: {mad_offset:.3f} м")
    
    # LOO
    print(f"\n  LOO-валидация:")
    loo_residuals = []
    for i in range(len(offsets)):
        rest = np.delete(offsets, i)
        pred = float(np.median(rest))
        resid = float(offsets[i] - pred)
        loo_residuals.append({
            "name": names[i],
            "predicted_offset": round(pred, 3),
            "actual_offset": round(float(offsets[i]), 3),
            "residual_m": round(resid, 3),
        })
        print(f"    {names[i]:<8}: pred={pred:+.3f}, actual={offsets[i]:+.3f}, "
              f"residual={resid:+.3f}")
    
    loo_rms = float(np.sqrt(np.mean(
        [r["residual_m"]**2 for r in loo_residuals])))
    loo_max = float(max(abs(r["residual_m"]) for r in loo_residuals))
    
    print(f"\n  LOO RMS:  {loo_rms:.3f} м")
    print(f"  LOO max:  {loo_max:.3f} м")
    
    if loo_rms > 0.5:
        print(f"\n  ⚠ LOO RMS > 0.5 м — возможно несогласованность точек")
        print(f"     посмотрите residuals внимательно перед использованием")
    elif loo_rms > 0.3:
        print(f"\n  ~ LOO RMS = 0.3-0.5 м — приемлемо, но не отлично")
    else:
        print(f"\n  ✓ LOO RMS ≤ 0.3 м — отличная калибровка")
    
    return {
        "n_flat_points": len(flat),
        "n_total_passport": len(passport_results),
        "flat_point_names": names,
        "offsets": [round(float(o), 3) for o in offsets],
        "median_offset_m": round(median_offset, 4),
        "mad_offset_m": round(mad_offset, 4),
        "loo_rms_m": round(loo_rms, 4),
        "loo_max_abs_m": round(loo_max, 4),
        "loo_residuals": loo_residuals,
    }


# =============================================================================
#                                   MAIN
# =============================================================================

def main() -> None:
    if not DTM_ELL.exists():
        sys.exit(f"Нет {DTM_ELL} — сначала запустите 2_smrf_filter.py")
    if not ATL08_GPKG.exists():
        sys.exit(f"Нет {ATL08_GPKG}")
    
    print("=" * 70)
    print("  [3/3] КАЛИБРОВКА DTM эллипсоид → БС-77")
    print("=" * 70)
    
    print(f"\nЗагрузка ATL08 ground точек...")
    atl08 = gpd.read_file(ATL08_GPKG, layer="atl08_with_dem")
    # x_m/y_m уже есть из шага 0
    if "x_m" not in atl08.columns:
        atl08_m = atl08.to_crs(WORK_EPSG)
        atl08["x_m"] = atl08_m.geometry.x
        atl08["y_m"] = atl08_m.geometry.y
    print(f"  загружено: {len(atl08):,}")
    
    # === ШАГ 1: пересэмплировать на DTM (не DSM)
    print(f"\n[1] Пересэмпл ATL08 на DTM (был на DSM)...")
    atl08_dtm = resample_atl08_on_dtm(DTM_ELL, atl08)
    print(f"  валидных после ресэмплинга и фильтрации |dZ|<5: {len(atl08_dtm):,}")
    
    # === ШАГ 2: коррегистрация — только сдвиг
    print(f"\n[2] Коррегистрация DTM по ICESat-2 (только константа)")
    coreg_summary = coregister_constant_shift(atl08_dtm)
    
    # Применяем сдвиг в эллипсоиде ещё раз (на DTM)
    # Это «остаточный» сдвиг после SMRF — обычно небольшой
    additional_shift_to_atl = -coreg_summary["median_dZ_m"]
    # Но не применяем сразу — будем интерпретировать как часть полного offset.
    # Финальный shift = median_dZ_atl + (offset_passport_after_atl_shift)
    # Это эквивалентно: shift = passport - DTM_ell, без двойного применения.
    
    # === ШАГ 3: оценка плоскостности паспортных точек
    print(f"\n[3] Анализ плоскостности паспортных точек:")
    passport_results = assess_passport_points(DTM_ELL, atl08_dtm)
    
    # === ШАГ 4: финальный сдвиг и LOO
    print(f"\n[4] Финальный сдвиг и LOO-валидация")
    cal_summary = calibrate_constant_offset(passport_results)
    
    # === ШАГ 5: применить к растру
    final_shift = cal_summary["median_offset_m"]
    print(f"\n[5] Применяем сдвиг {final_shift:+.3f} м к DTM → DTM_BS77")
    apply_constant_shift(DTM_ELL, DTM_BS77, final_shift)
    print(f"  ✓ {DTM_BS77}")
    
    # === ОТЧЁТ И ВИЗУАЛИЗАЦИЯ
    report = {
        "icesat2_coregister_diagnostic": coreg_summary,
        "passport_assessment": passport_results,
        "final_calibration": cal_summary,
        "final_shift_ellipsoid_to_bs77_m": final_shift,
        "params": {
            "win_px": WIN_PX,
            "pass_dem_std_max_m": PASS_DEM_STD_MAX_M,
            "pass_icesat_mad_max_m": PASS_ICESAT_MAD_MAX_M,
            "pass_icesat_min_pts": PASS_ICESAT_MIN_PTS,
        },
    }
    OUT_REPORT.write_text(json.dumps(report, indent=2, ensure_ascii=False,
                                      default=lambda o: float(o) if hasattr(o, "item") else str(o)))
    print(f"\n  Отчёт: {OUT_REPORT}")
    
    # PNG: scatter passport offsets с цветом по «плоский/нет»
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    # Левый: bar chart of offset by point
    ax = axes[0]
    flat_offsets = []
    flat_names = []
    rough_offsets = []
    rough_names = []
    for p in passport_results:
        if p.get("passes_filter"):
            flat_offsets.append(p["observed_offset_bs77_minus_dtm_ell"])
            flat_names.append(p["name"])
        elif p.get("dtm_ell_center") is not None:
            rough_offsets.append(p["h_bs77"] - p["dtm_ell_center"])
            rough_names.append(p["name"])
    
    if rough_offsets:
        ax.bar(rough_names, rough_offsets, color="gray", alpha=0.4,
               label="не плоские")
    if flat_offsets:
        ax.bar(flat_names, flat_offsets, color="green", label="плоские (используются)")
    ax.axhline(final_shift, color="red", linestyle="--",
                label=f"final shift = {final_shift:+.2f} м")
    ax.set_xlabel("Точка")
    ax.set_ylabel("h_BS77 - DTM_ell (м)")
    ax.set_title(f"Сдвиги по точкам. LOO RMS = {cal_summary['loo_rms_m']} м")
    ax.legend(fontsize=8)
    ax.tick_params(axis='x', rotation=45)
    ax.grid(alpha=0.3)
    
    # Правый: карта
    ax = axes[1]
    cmap = plt.cm.coolwarm
    for p in passport_results:
        if p.get("dtm_ell_center") is None:
            continue
        x, y = p["x_m"], p["y_m"]
        if p.get("passes_filter"):
            offset = p["observed_offset_bs77_minus_dtm_ell"]
            color = cmap((offset - final_shift + 1) / 2)
            ax.scatter(x, y, c=[color], s=200, marker="o",
                        edgecolors="black", linewidths=1.5)
        else:
            ax.scatter(x, y, c="lightgray", s=120, marker="x",
                        edgecolors="darkgray", linewidths=1.5)
        ax.annotate(p["name"], (x, y), fontsize=7,
                     xytext=(7, 7), textcoords="offset points")
    ax.set_xlabel("X UTM (м)")
    ax.set_ylabel("Y UTM (м)")
    ax.set_aspect("equal")
    ax.set_title("Карта точек: ○ плоские, × не плоские")
    ax.grid(alpha=0.3)
    
    fig.suptitle(f"Калибровка DTM → БС-77, shift = {final_shift:+.3f} м",
                  fontsize=11)
    fig.tight_layout()
    fig.savefig(OUT_PNG, dpi=130, bbox_inches="tight")
    plt.close(fig)
    print(f"  PNG:    {OUT_PNG}")
    
    print(f"\n{'═' * 70}")
    print(f"  ИТОГ")
    print(f"{'═' * 70}")
    print(f"  ICESat-2 на DTM:        median dZ = {coreg_summary['median_dZ_m']:+.3f}, "
          f"MAD = {coreg_summary['mad_dZ_m']:.3f} м")
    print(f"  Плоских точек:           {cal_summary['n_flat_points']}/{cal_summary['n_total_passport']}")
    print(f"  Финальный сдвиг:        {final_shift:+.3f} м")
    print(f"  LOO RMS:                {cal_summary['loo_rms_m']} м")
    print(f"\n  → DTM в БС-77: {DTM_BS77}")


if __name__ == "__main__":
    main()

  [3/3] КАЛИБРОВКА DTM эллипсоид → БС-77

Загрузка ATL08 ground точек...
  загружено: 43,334

[1] Пересэмпл ATL08 на DTM (был на DSM)...
  валидных после ресэмплинга и фильтрации |dZ|<5: 41,879

[2] Коррегистрация DTM по ICESat-2 (только константа)
  ATL08 ground vs DTM_ell:
    n = 41,879 точек
    median dZ = +0.220 м
    MAD       = 0.708 м
    95% CI median = [+0.216, +0.225] м

[3] Анализ плоскостности паспортных точек:

  name       h_BS77   DTM_ell  std_DTM ATL_n ATL_MAD    verdict
  Gorny       +2.85   +18.571    2.567     2     n/a    ✗ rough
  KZS_01      +4.35   +18.010    0.176    12    0.23     ✓ flat
  KZS_02      +3.37   +16.829    0.282    16    0.02     ✓ flat
  KZS_03      +3.77   +15.634    0.121    22    0.20     ✓ flat
  P01         +3.00   +14.790    0.686    16    0.27    ✗ rough
  P02         +4.91   +17.774    1.230     8    2.04    ✗ rough
  P03         +4.87   +16.625    1.091     7    1.13    ✗ rough
  P04         +3.00   +14.610    0.777    17    0.27    ✗ 

/tmp/ipykernel_55/3524698735.py:436: UserWarning: You passed a edgecolor/edgecolors ('darkgray') for an unfilled marker ('x').  Matplotlib is ignoring the edgecolor in favor of the facecolor.  This behavior may change in the future.
  ax.scatter(x, y, c="lightgray", s=120, marker="x",


  PNG:    spb_dem/calibration_v3.png

══════════════════════════════════════════════════════════════════════
  ИТОГ
══════════════════════════════════════════════════════════════════════
  ICESat-2 на DTM:        median dZ = +0.220, MAD = 0.708 м
  Плоских точек:           4/15
  Финальный сдвиг:        -13.345 м
  LOO RMS:                0.8447 м

  → DTM в БС-77: spb_dem/dtm_bs77.tif
